# Trabalho desenvolvido no Âmbito da disciplina [Aprendizagem Computacional Avançada] por José Cunha e Marta Antunes

# Estrutura do Notebook

Este notebook está organizado nas seguintes etapas:

1. **Desenvolvimento de um modelo de difusão base**  
   Implementação inicial de um modelo de difusão com parâmetros padrão.

2. **Aprimoramento manual da estrutura do modelo**  
   Ajustes manuais na arquitetura e nos hiperparâmetros com base em observações preliminares e conhecimento empírico.

3. **Grid Search de Hiperparâmetros**  
   Exploração sistemática dos seguintes hiperparâmetros para otimização do desempenho do modelo:
   
   ```python
   {
       'lr': [1e-4, 2e-4],
       'batch_size': [64, 128],
       'epochs': [30],
       'noise_steps': [1000],
       'base_channels': [32, 64],
       'dropout': [0.0, 0.1],
       'schedule_type': ['cosine'],
       'use_attention': [True],
       'channel_multipliers': [[1, 2, 4, 4]],
       'weight_decay': [1e-4]
   }
````

4. **Treino completo do modelo final**
   Treinamento do modelo com os melhores hiperparâmetros encontrados na etapa anterior, visando o melhor desempenho possível na tarefa definida.

```
```



# Importação das Bibliotecas

Nesta seção, são importadas todas as bibliotecas necessárias para o desenvolvimento e execução do modelo de difusão.

---

In [ ]:
# --- Install required libraries ---
!pip install torch torchvision matplotlib medmnist pytorch-fid --quiet
!pip install 'torchmetrics[image]'
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torch.utils.data import ConcatDataset

import matplotlib.pyplot as plt
from torchvision.utils import make_grid
import torchvision.transforms.functional as F

import medmnist
from medmnist import INFO, Evaluator

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 961.5/961.5 kB 25.3 MB/s eta 0:00:0

# 1. Desenvolvimento de um modelo de difusão base

## Implementação Inicial

O modelo de difusão base foi implementado com os seguintes componentes principais:

### Arquitetura do Modelo
- **U-Net** como backbone principal:
  - 3 camadas downsampling (64 → 128 → 256 canais)
  - Camada bottleneck com 256 canais
  - 3 camadas upsampling com conexões residuais
  - Normalização por grupos (GroupNorm)
  - Ativação GELU

### Processo de Difusão
- Escalonamento de ruído com agendamento cosseno (1000 passos)
- Função de perda MSE entre o ruído predito e o real
- Amostragem DDIM para geração acelerada

### Configurações de Treino
- Otimizador: Adam (lr=2e-4)
- Batch size: 128
- Número de épocas: 100
- Tamanho da imagem: 28x28 (BloodMNIST)

## Características Principais

1. **Agendamento de Variância Cosseno**:
   ```python
   def _cosine_beta_schedule(self, s=0.008):
       steps = self.noise_steps + 1
       x = torch.linspace(0, self.noise_steps, steps)
       alphas_cumprod = torch.cos(((x / self.noise_steps) + s) / (1 + s) * pi / 2) ** 2
       alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
       betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
       return torch.clip(betas, 0, 0.999)
   ```

2. **Mecanismo de Codificação Temporal**:
   - Codificação posicional sinusoidal
   - Projetada para a dimensão dos canais de cada bloco

3. **Amostragem Acelerada (DDIM)**:
   - Redução de 1000 para ~20 passos de difusão
   - Mantém qualidade comparável à amostragem completa

## Fluxo de Treinamento

1. Para cada batch de imagens:
   - Amostra um passo de tempo t uniformemente
   - Adiciona ruído às imagens de acordo com t
   - Prediz o ruído adicionado
   - Calcula a perda MSE entre o ruído real e predito
   - Atualiza os pesos via backpropagation

2. Métricas:
   - Loss de treino monitorada por época
   - Geração de amostras periódicas para avaliação visual
   - Cálculo de FID para avaliação quantitativa


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, utils
from torchvision.utils import save_image
from medmnist import BloodMNIST
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os
import time
from torch.utils.tensorboard import SummaryWriter
from math import sqrt, cos, pi
from scipy.linalg import sqrtm
from torchvision.models import inception_v3
import shutil
from google.colab import drive

# Mount Google Drive
try:
    drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/diffusion_results'
    print("Google Drive mounted successfully!")
except Exception as e:
    print(f"Error mounting Google Drive: {e}")
    DRIVE_PATH = './diffusion_results'

# Basic configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
timestamp = time.strftime("%Y%m%d-%H%M%S")

# Create project directory
diffusion_path = f'diffusion_model_{timestamp}'
os.makedirs(diffusion_path, exist_ok=True)

# Subdirectories
diffusion_model_dir = os.path.join(diffusion_path, 'models')
diffusion_images_dir = os.path.join(diffusion_path, 'generated_images')
diffusion_logs_dir = os.path.join(diffusion_path, 'logs')
diffusion_plots_dir = os.path.join(diffusion_path, 'plots')
fid_results_dir = os.path.join(diffusion_path, 'fid_results')

for dir_path in [diffusion_model_dir, diffusion_images_dir, diffusion_logs_dir, diffusion_plots_dir, fid_results_dir]:
    os.makedirs(dir_path, exist_ok=True)

# TensorBoard writer
diffusion_writer = SummaryWriter(diffusion_logs_dir)

# Dataset transformations
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

# Load dataset
train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
val_dataset = BloodMNIST(split='val', transform=data_transform, download=True)
test_dataset = BloodMNIST(split='test', transform=data_transform, download=True)
full_dataset = torch.utils.data.ConcatDataset([train_dataset, val_dataset, test_dataset])

batch_size = 128
dataloader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# ==============================================
# 1. Diffusion Process Definition
# ==============================================

class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=28, device="cuda"):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size
        self.device = device

        # Cosine variance schedule
        self.beta = self._cosine_beta_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def _cosine_beta_schedule(self, s=0.008):
        """Cosine variance schedule"""
        steps = self.noise_steps + 1
        x = torch.linspace(0, self.noise_steps, steps)
        alphas_cumprod = torch.cos(((x / self.noise_steps) + s) / (1 + s) * pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0, 0.999)

    def noise_images(self, x, t):
        """Add noise to images at timestep t"""
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

    def sample_timesteps(self, n):
        """Sample random timesteps for training"""
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model, n, ddim_steps=50):
        """Sample images from model with accelerated DDIM"""
        model.eval()
        with torch.no_grad():
            # DDIM sampling - use fewer steps
            if ddim_steps < self.noise_steps:
                step_size = self.noise_steps // ddim_steps
                timesteps = list(range(0, self.noise_steps, step_size))[:ddim_steps]
                timesteps = list(reversed(timesteps))
            else:
                timesteps = list(reversed(range(1, self.noise_steps)))

            x = torch.randn((n, 3, self.img_size, self.img_size)).to(self.device)

            for i, timestep in enumerate(tqdm(timesteps, desc="Sampling")):
                t = torch.full((n,), timestep, dtype=torch.long, device=self.device)

                # Predict noise
                predicted_noise = model(x, t)

                # DDIM update
                alpha_t = self.alpha_hat[timestep]
                if i < len(timesteps) - 1:
                    alpha_prev = self.alpha_hat[timesteps[i + 1]]
                else:
                    alpha_prev = torch.tensor(1.0).to(self.device)

                pred_x0 = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
                pred_x0 = torch.clamp(pred_x0, -1, 1)

                dir_xt = torch.sqrt(1 - alpha_prev) * predicted_noise
                x = torch.sqrt(alpha_prev) * pred_x0 + dir_xt

        model.train()
        x = (x.clamp(-1, 1) + 1) / 2  # Scale to [0,1]
        x = (x * 255).type(torch.uint8)
        return x

    def sample_fast(self, model, n, steps=20):
        """Faster sampling with fewer steps"""
        return self.sample(model, n, ddim_steps=steps)

# ==============================================
# 2. U-Net Architecture
# ==============================================

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(1, out_channels),
        )

    def forward(self, x):
        if self.residual:
            return F.gelu(x + self.double_conv(x))
        else:
            return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, t):
        x = self.maxpool_conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256):
        super().__init__()

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = nn.Sequential(
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, skip_x, t):
        x = self.up(x)
        diffY = skip_x.size()[2] - x.size()[2]
        diffX = skip_x.size()[3] - x.size()[3]

        x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                      diffY // 2, diffY - diffY // 2])

        x = torch.cat([skip_x, x], dim=1)
        x = self.conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class UNet(nn.Module):
    def __init__(self, c_in=3, c_out=3, time_dim=256, device="cuda"):
        super().__init__()
        self.time_dim = time_dim
        self.device = device

        # Encoder
        self.inc = DoubleConv(c_in, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 256)

        # Bottleneck
        self.bot1 = DoubleConv(256, 512)
        self.bot2 = DoubleConv(512, 512)
        self.bot3 = DoubleConv(512, 256)

        # Decoder
        self.up1 = Up(512, 128)
        self.up2 = Up(256, 64)
        self.up3 = Up(128, 64)
        self.outc = nn.Conv2d(64, c_out, kernel_size=1)

    def pos_encoding(self, t, channels):
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, channels, 2, device=self.device).float() / channels)
        )
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc

    def forward(self, x, t):
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)

        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1, t)
        x3 = self.down2(x2, t)
        x4 = self.down3(x3, t)

        # Bottleneck
        x4 = self.bot1(x4)
        x4 = self.bot2(x4)
        x4 = self.bot3(x4)

        # Decoder
        x = self.up1(x4, x3, t)
        x = self.up2(x, x2, t)
        x = self.up3(x, x1, t)
        output = self.outc(x)
        return output

# ==============================================
# 3. Fixed FID Calculation
# ==============================================

class InceptionV3Features(nn.Module):
    def __init__(self):
        super().__init__()
        # Load standard InceptionV3
        inception = inception_v3(pretrained=True)
        inception.eval()

        # Define feature extraction layers
        self.features = nn.Sequential(
            inception.Conv2d_1a_3x3,
            inception.Conv2d_2a_3x3,
            inception.Conv2d_2b_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Conv2d_3b_1x1,
            inception.Conv2d_4a_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Mixed_5b,
            inception.Mixed_5c,
            inception.Mixed_5d,
            inception.Mixed_6a,
            inception.Mixed_6b,
            inception.Mixed_6c,
            inception.Mixed_6d,
            inception.Mixed_6e,
            inception.Mixed_7a,
            inception.Mixed_7b,
            inception.Mixed_7c,
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        # Resize and normalize for InceptionV3
        x = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)

        # Convert from [-1,1] to [0,1] then to [0,255]
        x = (x + 1) / 2  # [0,1]
        x = x * 255      # [0,255]

        # Normalize using ImageNet stats
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device) * 255
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device) * 255
        x = (x - mean) / std

        # Handle grayscale by repeating channels
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)

        with torch.no_grad():
            features = self.features(x)
            features = features.view(features.size(0), -1)
        return features

def calculate_fid(real_features, fake_features):
    """Calculate FID between real and generated features"""
    mu_real = np.mean(real_features, axis=0)
    mu_fake = np.mean(fake_features, axis=0)

    sigma_real = np.cov(real_features, rowvar=False)
    sigma_fake = np.cov(fake_features, rowvar=False)

    # Add small epsilon for numerical stability
    eps = 1e-6
    sigma_real += eps * np.eye(sigma_real.shape[0])
    sigma_fake += eps * np.eye(sigma_fake.shape[0])

    diff = mu_real - mu_fake
    covmean = sqrtm(sigma_real.dot(sigma_fake))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

def extract_features_from_dataset(dataset, feature_extractor, batch_size=64, max_samples=None):
    """Extract features from dataset"""
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    features = []

    feature_extractor.eval()
    count = 0

    with torch.no_grad():
        for images, _ in tqdm(dataloader, desc="Extracting real features"):
            if max_samples and count >= max_samples:
                break

            images = images.to(device)
            batch_features = feature_extractor(images)
            features.append(batch_features.cpu().numpy())
            count += images.size(0)

    return np.concatenate(features, axis=0)

def extract_features_from_generated(generated_images, feature_extractor):
    """Extract features from generated images"""
    features = []
    feature_extractor.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(generated_images), 64), desc="Extracting generated features"):
            batch = generated_images[i:i+64].float().to(device)
            batch_features = feature_extractor(batch)
            features.append(batch_features.cpu().numpy())

    return np.concatenate(features, axis=0)

# ==============================================
# 4. Training Functions
# ==============================================

def train_diffusion():
    # Hyperparameters
    noise_steps = 1000
    lr = 2e-4
    epochs = 100
    img_size = 28

    # Initialize model and diffusion
    model = UNet(device=device).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    diffusion = Diffusion(noise_steps=noise_steps, img_size=img_size, device=device)

    best_loss = float('inf')
    train_losses = []

    print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch_idx, (images, _) in enumerate(progress_bar):
            images = images.to(device)

            # Sample timesteps
            t = diffusion.sample_timesteps(images.shape[0]).to(device)

            # Add noise
            x_t, noise = diffusion.noise_images(images, t)

            # Predict noise
            predicted_noise = model(x_t, t)

            # Loss
            loss = F.mse_loss(predicted_noise, noise)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)
        diffusion_writer.add_scalar('Loss/train', avg_loss, epoch)

        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.6f}")

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
                'train_losses': train_losses,
            }, os.path.join(diffusion_model_dir, 'best_diffusion_model.pth'))

        # Sample images periodically
        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            print(f"Generating samples at epoch {epoch+1}...")
            samples = diffusion.sample(model, n=16)
            grid = utils.make_grid(samples, nrow=4, padding=2, normalize=False)
            diffusion_writer.add_image('Generated Images', grid, epoch)

            sample_dir = os.path.join(diffusion_images_dir, f'epoch_{epoch+1}')
            os.makedirs(sample_dir, exist_ok=True)
            save_image(grid.float()/255, os.path.join(sample_dir, 'grid.png'))

    # Plot training loss
    loss_plot_path = os.path.join(diffusion_plots_dir, 'training_loss.png')
    plt.figure(figsize=(12, 8))
    plt.plot(train_losses, 'b-', linewidth=2, label='Training Loss')
    plt.title('Training Loss Over Time', fontsize=16)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(loss_plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    diffusion_writer.close()
    return model, diffusion, train_losses

# ==============================================
# 5. Generate Images and Calculate FID
# ==============================================

def generate_and_evaluate_fid(model, diffusion, num_samples=10000, num_fid_runs=5):
    """Generate images and calculate FID"""
    print(f"\nGenerating {num_samples} images...")

    batch_size = 500
    all_generated_images = []

    model.eval()
    torch.cuda.empty_cache()

    # Use fast sampling
    for i in tqdm(range(0, num_samples, batch_size), desc="Generating images"):
        current_batch_size = min(batch_size, num_samples - i)
        samples = diffusion.sample_fast(model, n=current_batch_size, steps=20)

        # Convert to float and scale to [0,1]
        samples = samples.float() / 255.0
        all_generated_images.append(samples.cpu())
        torch.cuda.empty_cache()

    all_generated_images = torch.cat(all_generated_images, dim=0)
    print(f"Total generated images: {len(all_generated_images)}")

    # Save sample images
    sample_dir = os.path.join(diffusion_images_dir, 'final_10k_samples')
    os.makedirs(sample_dir, exist_ok=True)
    grid = utils.make_grid(all_generated_images[:64], nrow=8, padding=2)
    save_image(grid, os.path.join(sample_dir, 'sample_grid_64.png'))

    print("\nCalculating FID...")

    # Prepare feature extractor
    feature_extractor = InceptionV3Features().to(device)

    # Extract features (use subset for speed)
    print("Extracting real features...")
    real_features = extract_features_from_dataset(
        full_dataset, feature_extractor, max_samples=5000
    )

    print("Extracting generated features...")
    fake_features = extract_features_from_generated(
        all_generated_images[:5000], feature_extractor
    )

    # Calculate FID multiple times
    fid_scores = []
    print(f"\nCalculating FID {num_fid_runs} times...")

    for run in range(num_fid_runs):
        print(f"Run {run + 1}/{num_fid_runs}")

        # Use smaller subsets for efficiency
        sample_size = min(2000, len(real_features), len(fake_features))
        real_idx = np.random.choice(len(real_features), sample_size, replace=False)
        fake_idx = np.random.choice(len(fake_features), sample_size, replace=False)

        real_sub = real_features[real_idx]
        fake_sub = fake_features[fake_idx]

        fid_score = calculate_fid(real_sub, fake_sub)
        fid_scores.append(fid_score)
        print(f"FID Score (Run {run + 1}): {fid_score:.4f}")

    # Calculate statistics
    fid_mean = np.mean(fid_scores)
    fid_std = np.std(fid_scores)

    print(f"\n{'='*50}")
    print(f"FINAL FID RESULTS:")
    print(f"Mean FID: {fid_mean:.4f} ± {fid_std:.4f}")
    print(f"Individual FID Scores: {[f'{score:.4f}' for score in fid_scores]}")
    print(f"{'='*50}")

    # Save results
    fid_results = {
        'fid_scores': fid_scores,
        'fid_mean': fid_mean,
        'fid_std': fid_std,
        'num_samples': num_samples,
        'num_runs': num_fid_runs,
    }

    import json
    with open(os.path.join(fid_results_dir, 'fid_results.json'), 'w') as f:
        json.dump(fid_results, f, indent=2)

    return fid_results, all_generated_images

# ==============================================
# 6. Save to Google Drive
# ==============================================

def save_to_drive(source_path, drive_path):
    """Copy files to Google Drive"""
    try:
        os.makedirs(drive_path, exist_ok=True)
        if os.path.isdir(source_path):
            shutil.copytree(source_path, os.path.join(drive_path, os.path.basename(source_path)),
                          dirs_exist_ok=True)
        else:
            shutil.copy2(source_path, drive_path)
        print(f"Files saved to Google Drive: {drive_path}")
    except Exception as e:
        print(f"Error saving to Google Drive: {e}")

# ==============================================
# 7. Main Execution
# ==============================================

if __name__ == "__main__":
    print("Starting diffusion model training...")
    print(f"Using device: {device}")

    # Train model
    model, diffusion, train_losses = train_diffusion()

    print("\n" + "="*70)
    print("TRAINING COMPLETED!")
    print("="*70)

    # Generate images and calculate FID
    print("\nStarting image generation and FID calculation...")
    fid_results, generated_images = generate_and_evaluate_fid(
        model, diffusion, num_samples=10000, num_fid_runs=5
    )

    print("\n" + "="*70)
    print("EVALUATION COMPLETED!")
    print("="*70)

    # Create summary
    summary_path = os.path.join(diffusion_path, 'experiment_summary.txt')
    with open(summary_path, 'w') as f:
        f.write(f"Mean FID: {fid_results['fid_mean']:.4f} ± {fid_results['fid_std']:.4f}\n")
        f.write(f"Best loss: {min(train_losses):.6f}\n")

    # Save to Google Drive
    print("\nSaving results to Google Drive...")
    save_to_drive(diffusion_path, DRIVE_PATH)

    print("\n🎉 EXPERIMENT COMPLETED SUCCESSFULLY! 🎉")
    print(f"📊 FID Score: {fid_results['fid_mean']:.4f} ± {fid_results['fid_std']:.4f}")
    print(f"📁 Results saved in: {diffusion_path}")
    if DRIVE_PATH != './diffusion_results':
        print(f"📁 Backup in Google Drive: {DRIVE_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!
Starting diffusion model training...
Using device: cuda
Model created with 21,369,603 parameters


Epoch 1/100: 100%|██████████| 134/134 [00:05<00:00, 24.30it/s, loss=0.108]


Epoch 1/100, Average Loss: 0.272840


Epoch 2/100: 100%|██████████| 134/134 [00:05<00:00, 24.58it/s, loss=0.0849]


Epoch 2/100, Average Loss: 0.103020


...

Total generated images: 10000

Calculating FID...
Extracting real features...


Extracting real features:  29%|██▉       | 79/268 [00:02<00:06, 29.89it/s]


Extracting generated features...


Extracting generated features: 100%|██████████| 79/79 [00:02<00:00, 33.10it/s]



Calculating FID 5 times...
Run 1/5
FID Score (Run 1): 134.1919
Run 2/5
FID Score (Run 2): 131.8685
Run 3/5
FID Score (Run 3): 134.4253
Run 4/5
FID Score (Run 4): 135.6175
Run 5/5
FID Score (Run 5): 135.1981

FINAL FID RESULTS:
Mean FID: 134.2603 ± 1.3019
Individual FID Scores: ['134.1919', '131.8685', '134.4253', '135.6175', '135.1981']

EVALUATION COMPLETED!

Saving results to Google Drive...
Files saved to Google Drive: /content/drive/MyDrive/diffusion_results

🎉 EXPERIMENT COMPLETED SUCCESSFULLY! 🎉
📊 FID Score: 134.2603 ± 1.3019
📁 Results saved in: diffusion_model_20250526-231146
📁 Backup in Google Drive: /content/drive/MyDrive/diffusion_results


# Ponto 2: Aprimoramento Manual da Estrutura do Modelo

Este notebook implementa melhorias manuais na arquitetura do modelo de difusão baseadas em observações preliminares e conhecimento empírico sobre modelos de difusão de alta qualidade.

## Principais Melhorias Implementadas

### 1. **Arquitetura U-Net Aprimorada (EnhancedUNet)**
- **Blocos de Atenção**: Implementação de `AttentionBlock` no bottleneck para capturar dependências de longo alcance
- **Normalização em Grupos**: Uso de `GroupNorm` com 8 grupos para melhor estabilidade de treinamento
- **Conexões Residuais Melhoradas**: Ajuste automático de canais em conexões residuais
- **Dimensões de Canal Corrigidas**: Cálculo preciso das dimensões de entrada/saída em cada camada

### 2. **Processo de Difusão Otimizado**
- **Schedule de Ruído Melhorado**: Implementação de `_improved_cosine_schedule` com melhor distribuição de ruído
- **Amostragem DDIM Acelerada**: Método `sample_fast` com 20 passos para geração rápida
- **Clipping Aprimorado**: Melhor controle dos valores durante a amostragem

### 3. **Otimizações de Treinamento**
- **Otimizador AdamW**: Uso do AdamW com weight decay (1e-4) para melhor regularização
- **Scheduler Cosine**: `CosineAnnealingLR` para decaimento suave da taxa de aprendizado
- **Gradient Clipping**: Limitação dos gradientes a 1.0 para estabilidade
- **Épocas Aumentadas**: 150 épocas em vez de 100 para melhor convergência

### 4. **Melhorias na Avaliação**
- **Geração Robusta**: Tratamento de erros durante a geração de imagens
- **FID Calculation Otimizada**: Uso de subconjuntos menores (1000 amostras) para eficiência
- **Logging Avançado**: Registro da taxa de aprendizado no TensorBoard

## Estrutura do Código

```python
# Componentes principais:
class AttentionBlock(nn.Module)      # Blocos de atenção para dependências de longo alcance
class DoubleConv(nn.Module)          # Convolução dupla com residual melhorado
class Down(nn.Module)                # Downsampling com embedding temporal
class Up(nn.Module)                  # Upsampling com skip connections
class EnhancedUNet(nn.Module)        # Arquitetura U-Net completa aprimorada
class Diffusion                      # Processo de difusão otimizado
```

## Hiperparâmetros Utilizados

| Parâmetro | Valor | Justificativa |
|-----------|-------|---------------|
| `lr` | 2e-4 | Taxa padrão estável para modelos de difusão |
| `epochs` | 150 | Mais épocas para melhor convergência |
| `batch_size` | 128 | Equilibrio entre performance e memória |
| `weight_decay` | 1e-4 | Regularização para evitar overfitting |
| `gradient_clip` | 1.0 | Estabilidade durante treinamento |
| `noise_steps` | 1000 | Padrão para modelos de difusão |
| `ddim_steps` | 20 | Amostragem rápida para avaliação |

## Arquitetura da Rede

### Encoder
- **inc**: DoubleConv(3 → 64)
- **down1**: Down(64 → 128) + time embedding
- **down2**: Down(128 → 256) + time embedding  
- **down3**: Down(256 → 256) + time embedding

### Bottleneck
- **bot1**: DoubleConv(256 → 512)
- **bot_attn**: AttentionBlock(512) - **Novidade!**
- **bot2**: DoubleConv(512 → 256)

### Decoder
- **up1**: Up(512 → 256) + skip connection
- **up2**: Up(384 → 128) + skip connection
- **up3**: Up(192 → 64) + skip connection
- **outc**: Conv2d(64 → 3)

## Melhorias Específicas Implementadas

### 1. **Bloco de Atenção**
```python
class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)
```

### 2. **Conexões Residuais Inteligentes**
```python
if self.residual and in_channels != out_channels:
    self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
else:
    self.residual_conv = nn.Identity()
```

### 3. **Schedule de Ruído Melhorado**
```python
def _improved_cosine_schedule(self, s=0.008):
    """Improved cosine variance schedule with better noise distribution"""
    steps = self.noise_steps + 1
    x = torch.linspace(0, self.noise_steps, steps)
    alphas_cumprod = torch.cos(((x / self.noise_steps) + s) / (1 + s) * pi / 2) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0, 0.999)
```

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, utils
from torchvision.utils import save_image
from medmnist import BloodMNIST
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os
import time
from torch.utils.tensorboard import SummaryWriter
from math import sqrt, cos, pi
from scipy.linalg import sqrtm
from torchvision.models import inception_v3
import shutil

# Basic configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
timestamp = time.strftime("%Y%m%d-%H%M%S")

# Create project directory
diffusion_path = f'diffusion_model_{timestamp}'
os.makedirs(diffusion_path, exist_ok=True)

# Subdirectories
diffusion_model_dir = os.path.join(diffusion_path, 'models')
diffusion_images_dir = os.path.join(diffusion_path, 'generated_images')
diffusion_logs_dir = os.path.join(diffusion_path, 'logs')
diffusion_plots_dir = os.path.join(diffusion_path, 'plots')
fid_results_dir = os.path.join(diffusion_path, 'fid_results')

for dir_path in [diffusion_model_dir, diffusion_images_dir, diffusion_logs_dir, diffusion_plots_dir, fid_results_dir]:
    os.makedirs(dir_path, exist_ok=True)

# TensorBoard writer
diffusion_writer = SummaryWriter(diffusion_logs_dir)

# Dataset transformations
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # Normalize to [-1, 1]
])

# Load dataset
train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
val_dataset = BloodMNIST(split='val', transform=data_transform, download=True)
test_dataset = BloodMNIST(split='test', transform=data_transform, download=True)
full_dataset = torch.utils.data.ConcatDataset([train_dataset, val_dataset, test_dataset])

batch_size = 128  # Reduced to avoid memory issues
dataloader = DataLoader(full_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

# ==============================================
# 1. Improved Diffusion Process
# ==============================================

class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02, img_size=28, device="cuda"):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size
        self.device = device

        # Improved cosine variance schedule
        self.beta = self._improved_cosine_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def _improved_cosine_schedule(self, s=0.008):
        """Improved cosine variance schedule with better noise distribution"""
        steps = self.noise_steps + 1
        x = torch.linspace(0, self.noise_steps, steps)
        alphas_cumprod = torch.cos(((x / self.noise_steps) + s) / (1 + s) * pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0, 0.999)

    def noise_images(self, x, t):
        """Add noise to images at timestep t"""
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

    def sample_timesteps(self, n):
        """Sample random timesteps for training"""
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model, n, ddim_steps=50):
        """Sample images from model with accelerated DDIM"""
        model.eval()
        with torch.no_grad():
            if ddim_steps < self.noise_steps:
                step_size = self.noise_steps // ddim_steps
                timesteps = list(range(0, self.noise_steps, step_size))[:ddim_steps]
                timesteps = list(reversed(timesteps))
            else:
                timesteps = list(reversed(range(1, self.noise_steps)))

            x = torch.randn((n, 3, self.img_size, self.img_size)).to(self.device)

            for i, timestep in enumerate(tqdm(timesteps, desc="Sampling", leave=False)):
                t = torch.full((n,), timestep, dtype=torch.long, device=self.device)
                predicted_noise = model(x, t)
                alpha_t = self.alpha_hat[timestep]
                if i < len(timesteps) - 1:
                    alpha_prev = self.alpha_hat[timesteps[i + 1]]
                else:
                    alpha_prev = torch.tensor(1.0).to(self.device)

                pred_x0 = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
                pred_x0 = torch.clamp(pred_x0, -1, 1)

                dir_xt = torch.sqrt(1 - alpha_prev) * predicted_noise
                x = torch.sqrt(alpha_prev) * pred_x0 + dir_xt

        model.train()
        x = (x.clamp(-1, 1) + 1) / 2
        x = (x * 255).type(torch.uint8)
        return x

    def sample_fast(self, model, n, steps=20):
        """Faster sampling with fewer steps"""
        return self.sample(model, n, ddim_steps=steps)

# ==============================================
# 2. Fixed U-Net Architecture
# ==============================================

class AttentionBlock(nn.Module):
    """Self-attention block for capturing long-range dependencies"""
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)

        # Reshape for attention
        q = q.view(B, C, H * W).transpose(1, 2)
        k = k.view(B, C, H * W)
        v = v.view(B, C, H * W).transpose(1, 2)

        # Attention computation
        scale = 1 / (C ** 0.5)
        attn = torch.softmax(torch.bmm(q, k) * scale, dim=-1)

        out = torch.bmm(attn, v).transpose(1, 2).view(B, C, H, W)
        out = self.proj(out) + x
        return out

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels

        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, mid_channels),
            nn.GELU(),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(8, out_channels),
        )

        if self.residual and in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        else:
            self.residual_conv = nn.Identity()

    def forward(self, x):
        if self.residual:
            return F.gelu(self.residual_conv(x) + self.double_conv(x))
        else:
            return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, t):
        x = self.maxpool_conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = nn.Sequential(
            DoubleConv(in_channels, in_channels, residual=True),
            DoubleConv(in_channels, out_channels, in_channels // 2),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, skip_x, t):
        x = self.up(x)
        diffY = skip_x.size()[2] - x.size()[2]
        diffX = skip_x.size()[3] - x.size()[3]

        x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                      diffY // 2, diffY - diffY // 2])
        x = torch.cat([skip_x, x], dim=1)
        x = self.conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class EnhancedUNet(nn.Module):
    def __init__(self, c_in=3, c_out=3, time_dim=256, device="cuda"):
        super().__init__()
        self.time_dim = time_dim
        self.device = device

        # Encoder
        self.inc = DoubleConv(c_in, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 256)

        # Bottleneck with attention - Fixed channel sizes
        self.bot1 = DoubleConv(256, 512)
        self.bot_attn = AttentionBlock(512)
        self.bot2 = DoubleConv(512, 256)

        # Decoder - Fixed channel calculations
        self.up1 = Up(512, 256)   # 256 + 256 = 512 input
        self.up2 = Up(384, 128)   # 256 + 128 = 384 input
        self.up3 = Up(192, 64)    # 128 + 64 = 192 input
        self.outc = nn.Conv2d(64, c_out, kernel_size=1)

    def pos_encoding(self, t, channels):
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, channels, 2, device=self.device).float() / channels)
        )
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc

    def forward(self, x, t):
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)

        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1, t)
        x3 = self.down2(x2, t)
        x4 = self.down3(x3, t)

        # Bottleneck
        x4 = self.bot1(x4)
        x4 = self.bot_attn(x4)
        x4 = self.bot2(x4)

        # Decoder
        x = self.up1(x4, x3, t)
        x = self.up2(x, x2, t)
        x = self.up3(x, x1, t)
        output = self.outc(x)
        return output

# ==============================================
# 3. Simplified FID Calculation (from second code)
# ==============================================

class InceptionV3Features(nn.Module):
    def __init__(self):
        super().__init__()
        # Load standard InceptionV3
        inception = inception_v3(pretrained=True)
        inception.eval()

        # Define feature extraction layers
        self.features = nn.Sequential(
            inception.Conv2d_1a_3x3,
            inception.Conv2d_2a_3x3,
            inception.Conv2d_2b_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Conv2d_3b_1x1,
            inception.Conv2d_4a_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Mixed_5b,
            inception.Mixed_5c,
            inception.Mixed_5d,
            inception.Mixed_6a,
            inception.Mixed_6b,
            inception.Mixed_6c,
            inception.Mixed_6d,
            inception.Mixed_6e,
            inception.Mixed_7a,
            inception.Mixed_7b,
            inception.Mixed_7c,
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        # Resize and normalize for InceptionV3
        x = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)

        # Convert from [-1,1] to [0,1] then to [0,255]
        x = (x + 1) / 2  # [0,1]
        x = x * 255      # [0,255]

        # Normalize using ImageNet stats
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device) * 255
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device) * 255
        x = (x - mean) / std

        # Handle grayscale by repeating channels
        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)

        with torch.no_grad():
            features = self.features(x)
            features = features.view(features.size(0), -1)
        return features

def calculate_fid(real_features, fake_features):
    """Calculate FID between real and generated features"""
    mu_real = np.mean(real_features, axis=0)
    mu_fake = np.mean(fake_features, axis=0)

    sigma_real = np.cov(real_features, rowvar=False)
    sigma_fake = np.cov(fake_features, rowvar=False)

    # Add small epsilon for numerical stability
    eps = 1e-6
    sigma_real += eps * np.eye(sigma_real.shape[0])
    sigma_fake += eps * np.eye(sigma_fake.shape[0])

    diff = mu_real - mu_fake
    covmean = sqrtm(sigma_real.dot(sigma_fake))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

def extract_features_from_dataset(dataset, feature_extractor, batch_size=64, max_samples=None):
    """Extract features from dataset"""
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    features = []

    feature_extractor.eval()
    count = 0

    with torch.no_grad():
        for images, _ in tqdm(dataloader, desc="Extracting real features"):
            if max_samples and count >= max_samples:
                break

            images = images.to(device)
            batch_features = feature_extractor(images)
            features.append(batch_features.cpu().numpy())
            count += images.size(0)

    return np.concatenate(features, axis=0)

def extract_features_from_generated(generated_images, feature_extractor):
    """Extract features from generated images"""
    features = []
    feature_extractor.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(generated_images), 64), desc="Extracting generated features"):
            batch = generated_images[i:i+64].float().to(device)
            batch_features = feature_extractor(batch)
            features.append(batch_features.cpu().numpy())

    return np.concatenate(features, axis=0)

# ==============================================
# 4. Training Functions
# ==============================================

def train_diffusion():
    # Hyperparameters
    noise_steps = 1000
    lr = 2e-4
    epochs = 150
    img_size = 28

    # Initialize model and diffusion
    model = EnhancedUNet(device=device).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    diffusion = Diffusion(noise_steps=noise_steps, img_size=img_size, device=device)

    best_loss = float('inf')
    train_losses = []

    print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")

        for batch_idx, (images, _) in enumerate(progress_bar):
            images = images.to(device)

            # Sample timesteps
            t = diffusion.sample_timesteps(images.shape[0]).to(device)

            # Add noise
            x_t, noise = diffusion.noise_images(images, t)

            # Predict noise
            predicted_noise = model(x_t, t)

            # Loss
            loss = F.mse_loss(predicted_noise, noise)

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item(), lr=scheduler.get_last_lr()[0])

        scheduler.step()
        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)
        diffusion_writer.add_scalar('Loss/train', avg_loss, epoch)
        diffusion_writer.add_scalar('Learning_Rate', scheduler.get_last_lr()[0], epoch)

        print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.6f}")

        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
                'train_losses': train_losses,
            }, os.path.join(diffusion_model_dir, 'best_diffusion_model.pth'))

        # Sample images periodically
        if (epoch + 1) % 20 == 0 or epoch == epochs - 1:
            print(f"Generating samples at epoch {epoch+1}...")
            samples = diffusion.sample(model, n=16, ddim_steps=20)
            grid = utils.make_grid(samples, nrow=4, padding=2, normalize=False)
            diffusion_writer.add_image('Generated Images', grid, epoch)

            sample_dir = os.path.join(diffusion_images_dir, f'epoch_{epoch+1}')
            os.makedirs(sample_dir, exist_ok=True)
            save_image(grid.float()/255, os.path.join(sample_dir, 'grid.png'))

    # Plot training loss
    loss_plot_path = os.path.join(diffusion_plots_dir, 'training_loss.png')
    plt.figure(figsize=(12, 8))
    plt.plot(train_losses, 'b-', linewidth=2, label='Training Loss')
    plt.title('Training Loss Over Time', fontsize=16)
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('Loss', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(loss_plot_path, dpi=300, bbox_inches='tight')
    plt.close()

    diffusion_writer.close()
    return model, diffusion, train_losses

# ==============================================
# 5. Generate Images and Calculate FID
# ==============================================

def generate_and_evaluate_fid(model, diffusion, num_samples=3000, num_fid_runs=5):
    """Generate images and calculate FID"""
    print(f"\nGenerating {num_samples} images...")

    batch_size = 500
    all_generated_images = []

    model.eval()
    torch.cuda.empty_cache()

    # Use fast sampling
    for i in tqdm(range(0, num_samples, batch_size), desc="Generating images"):
        current_batch_size = min(batch_size, num_samples - i)
        try:
            samples = diffusion.sample_fast(model, n=current_batch_size, steps=20)
            # Convert to float and scale to [0,1]
            samples = samples.float() / 255.0
            all_generated_images.append(samples.cpu())
            torch.cuda.empty_cache()
        except RuntimeError as e:
            print(f"Error generating batch {i}: {e}")
            torch.cuda.empty_cache()
            continue

    if not all_generated_images:
        raise RuntimeError("No images could be generated")

    all_generated_images = torch.cat(all_generated_images, dim=0)
    print(f"Total generated images: {len(all_generated_images)}")

    # Save sample images
    sample_dir = os.path.join(diffusion_images_dir, f'final_{len(all_generated_images)}_samples')
    os.makedirs(sample_dir, exist_ok=True)
    grid = utils.make_grid(all_generated_images[:64], nrow=8, padding=2)
    save_image(grid, os.path.join(sample_dir, 'sample_grid_64.png'))

    print("\nCalculating FID...")

    try:
        # Prepare feature extractor
        feature_extractor = InceptionV3Features().to(device)

        # Extract features (use subset for speed)
        print("Extracting real features...")
        real_features = extract_features_from_dataset(
            full_dataset, feature_extractor, max_samples=min(2000, len(all_generated_images))
        )

        print("Extracting generated features...")
        fake_features = extract_features_from_generated(
            all_generated_images[:min(2000, len(all_generated_images))], feature_extractor
        )

        # Calculate FID multiple times
        fid_scores = []
        print(f"\nCalculating FID {num_fid_runs} times...")

        for run in range(num_fid_runs):
            print(f"Run {run + 1}/{num_fid_runs}")

            # Use smaller subsets for efficiency
            sample_size = min(1000, len(real_features), len(fake_features))
            real_idx = np.random.choice(len(real_features), sample_size, replace=False)
            fake_idx = np.random.choice(len(fake_features), sample_size, replace=False)

            real_sub = real_features[real_idx]
            fake_sub = fake_features[fake_idx]

            fid_score = calculate_fid(real_sub, fake_sub)
            fid_scores.append(fid_score)
            print(f"FID Score (Run {run + 1}): {fid_score:.4f}")

        # Calculate statistics
        fid_mean = np.mean(fid_scores)
        fid_std = np.std(fid_scores)

        print(f"\n{'='*50}")
        print(f"FINAL FID RESULTS:")
        print(f"Mean FID: {fid_mean:.4f} ± {fid_std:.4f}")
        print(f"Individual FID Scores: {[f'{score:.4f}' for score in fid_scores]}")
        print(f"{'='*50}")

        # Save results
        fid_results = {
            'fid_scores': fid_scores,
            'fid_mean': fid_mean,
            'fid_std': fid_std,
            'num_samples': len(all_generated_images),
            'num_runs': num_fid_runs,
        }

    except Exception as e:
        print(f"Error calculating FID: {e}")
        print("Continuing without FID calculation...")
        fid_results = {
            'fid_scores': [],
            'fid_mean': -1,
            'fid_std': -1,
            'num_samples': len(all_generated_images),
            'num_runs': 0,
            'error': str(e)
        }

    import json
    with open(os.path.join(fid_results_dir, 'fid_results.json'), 'w') as f:
        json.dump(fid_results, f, indent=2)

    return fid_results, all_generated_images

# ==============================================
# 6. Main Execution
# ==============================================

if __name__ == "__main__":
    print("Starting diffusion model training...")
    print(f"Using device: {device}")
    print(f"Dataset size: {len(full_dataset)} images")

    # Train model
    model, diffusion, train_losses = train_diffusion()

    print("\n" + "="*70)
    print("TRAINING COMPLETED!")
    print("="*70)

    # Generate images and calculate FID
    print("\nStarting image generation and FID calculation...")
    fid_results, generated_images = generate_and_evaluate_fid(
        model, diffusion, num_samples=3000, num_fid_runs=5
    )

    print("\n" + "="*70)
    print("EVALUATION COMPLETED!")
    print("="*70)

    # Create summary
    summary_path = os.path.join(diffusion_path, 'experiment_summary.txt')
    with open(summary_path, 'w') as f:
        f.write(f"Diffusion Model Results\n")
        f.write(f"=====================\n")
        f.write(f"Mean FID: {fid_results['fid_mean']:.4f} ± {fid_results['fid_std']:.4f}\n")
        f.write(f"Best training loss: {min(train_losses):.6f}\n")
        f.write(f"Final training loss: {train_losses[-1]:.6f}\n")
        f.write(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}\n")
        f.write(f"Dataset size: {len(full_dataset)}\n")

    print("\n🎉 EXPERIMENT COMPLETED SUCCESSFULLY! 🎉")
    print(f"📊 FID Score: {fid_results['fid_mean']:.4f} ± {fid_results['fid_std']:.4f}")
    print(f"📁 Results saved in: {diffusion_path}")
    print(f"📈 Best training loss: {min(train_losses):.6f}")

Starting diffusion model training...
Using device: cuda
Dataset size: 17092 images
Model created with 21,438,083 parameters


Epoch 1/150: 100%|██████████| 134/134 [00:08<00:00, 15.93it/s, loss=0.0994, lr=0.0002]


Epoch 1/150, Average Loss: 0.214259


Epoch 2/150: 100%|██████████| 134/134 [00:08<00:00, 16.02it/s, loss=0.0944, lr=0.0002]


Epoch 2/150, Average Loss: 0.069985


...

Total generated images: 3000

Calculating FID...
Extracting real features...


Extracting real features:  12%|█▏        | 32/268 [00:01<00:08, 27.51it/s]


Extracting generated features...


Extracting generated features: 100%|██████████| 32/32 [00:00<00:00, 32.46it/s]



Calculating FID 5 times...
Run 1/5
FID Score (Run 1): 114.2819
Run 2/5
FID Score (Run 2): 114.0958
Run 3/5
FID Score (Run 3): 112.2090
Run 4/5
FID Score (Run 4): 113.8264
Run 5/5
FID Score (Run 5): 114.1846

FINAL FID RESULTS:
Mean FID: 113.7195 ± 0.7703
Individual FID Scores: ['114.2819', '114.0958', '112.2090', '113.8264', '114.1846']

EVALUATION COMPLETED!

🎉 EXPERIMENT COMPLETED SUCCESSFULLY! 🎉
📊 FID Score: 113.7195 ± 0.7703
📁 Results saved in: diffusion_model_20250527-122753
📈 Best training loss: 0.033788


# 3. Grid Search de Hiperparâmetros

## Visão Geral
Esta seção implementa uma busca em grade (grid search) sistemática para encontrar a combinação ótima de hiperparâmetros que maximize o desempenho do modelo de difusão. O processo é automatizado e avalia múltiplas configurações usando métricas quantitativas.

## Hiperparâmetros Explorados

### Configuração de Busca Rápida (Quick Mode)
Para testes e desenvolvimento inicial:

```python
param_grid_quick = {
    'lr': [1e-4, 2e-4],                    # Taxa de aprendizado
    'batch_size': [64, 128],               # Tamanho do lote
    'epochs': [30],                        # Número de épocas (treinamento rápido)
    'noise_steps': [1000],                 # Passos de ruído na difusão
    'base_channels': [32, 64],             # Canais base da U-Net
    'dropout': [0.0, 0.1],                 # Taxa de dropout
    'schedule_type': ['cosine'],           # Tipo de agendamento de variância
    'use_attention': [True],               # Uso de blocos de atenção
    'channel_multipliers': [[1, 2, 4, 4]], # Multiplicadores de canal
    'weight_decay': [1e-4]                 # Decaimento de peso
}
```

### Configuração de Busca Completa
Para exploração exaustiva com treinamento mais profundo:

```python
param_grid_full = {
    'lr': [5e-5, 1e-4, 2e-4, 3e-4],
    'batch_size': [64, 128, 256],
    'epochs': [80, 120, 150],              # Treinamento extensivo
    'noise_steps': [500, 1000, 1500],
    'base_channels': [32, 64, 96],
    'dropout': [0.0, 0.1, 0.2],
    'schedule_type': ['cosine', 'linear', 'quadratic'],
    'use_attention': [True, False],
    'channel_multipliers': [[1, 2, 4, 4], [1, 2, 4, 8], [1, 1, 2, 4]],
    'weight_decay': [1e-5, 1e-4, 5e-4]
}
```

## Processo de Treinamento

### Duração do Treinamento por Experimento
- **Modo Rápido**: 30 épocas fixas para avaliação preliminar
- **Modo Completo**: Entre 80-150 épocas dependendo da configuração
- **Early Stopping**: Monitoramento da loss para evitar overfitting
- **Learning Rate Scheduling**: CosineAnnealingLR aplicado durante todo o treinamento

### Monitoramento Durante Treinamento
```python
for epoch in range(epochs):
    epoch_loss = 0
    for batch_idx, (images, _) in enumerate(dataloader):
        # Forward pass e cálculo da loss
        t = diffusion.sample_timesteps(images.shape[0])
        x_t, noise = diffusion.noise_images(images, t)
        predicted_noise = model(x_t, t)
        loss = F.mse_loss(predicted_noise, noise)
        
        # Backpropagation com gradient clipping
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    
    # Atualização do scheduler
    scheduler.step()
    avg_loss = epoch_loss / len(dataloader)
    
    # Tracking da melhor loss para seleção posterior
    if avg_loss < best_loss:
        best_loss = avg_loss
```

## Processo de Avaliação e Seleção

### Métricas de Avaliação
1. **Loss de Treinamento**: MSE entre ruído predito e real
2. **FID Score**: Métrica principal para qualidade das imagens geradas
3. **Parâmetros do Modelo**: Para análise de complexidade

### Pipeline de Seleção de Modelos

#### 1. Avaliação Individual
Para cada configuração testada:
```python
# Após treinamento completo
model, diffusion, best_loss, train_losses = train_diffusion_with_params(config, dataset)

# Geração de amostras para FID
num_samples = 500  # Amostras para avaliação
samples = diffusion.sample(model, n=num_samples, ddim_steps=20)

# Cálculo do FID Score
fake_features = extract_features_from_generated(samples, feature_extractor)
fid_score = calculate_fid(real_features, fake_features)
```

#### 2. Critério de Seleção Principal
```python
# Ranking baseado em FID Score (menor é melhor)
if fid_score < best_fid:
    best_fid = fid_score
    best_config = config.copy()
    best_model = model
    best_diffusion = diffusion
    
    # Salvamento automático do melhor modelo
    torch.save({
        'config': config,
        'model_state_dict': model.state_dict(),
        'fid_score': fid_score,
        'best_loss': best_loss
    }, 'best_gridsearch_model.pth')
```

#### 3. Análise Comparativa
```python
# Ranking de todos os experimentos
successful_results.sort(key=lambda x: x['fid_score'])

# Top 10 melhores configurações
print("🏆 TOP 10 BEST FID SCORES:")
for i, result in enumerate(successful_results[:10]):
    print(f"{i+1}. FID: {result['fid_score']:6.2f} | "
          f"Loss: {result['best_loss']:.6f} | "
          f"Exp: {result['experiment']}")
```


## Análise de Resultados

### Visualizações Geradas
1. **Scatter Plot**: FID vs Loss de treinamento
2. **Gráficos de Barras**: Impacto de hiperparâmetros individuais  
3. **Histograma**: Distribuição dos scores FID
4. **Ranking**: Top 10 melhores experimentos


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, utils
from torchvision.utils import save_image
from medmnist import BloodMNIST
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os
import time
from torch.utils.tensorboard import SummaryWriter
from math import sqrt, cos, pi
from scipy.linalg import sqrtm
from torchvision.models import inception_v3
import shutil
import json
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Basic configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
timestamp = time.strftime("%Y%m%d-%H%M%S")

# Create project directory
diffusion_path = f'diffusion_gridsearch_{timestamp}'
os.makedirs(diffusion_path, exist_ok=True)

# Subdirectories
diffusion_model_dir = os.path.join(diffusion_path, 'models')
diffusion_images_dir = os.path.join(diffusion_path, 'generated_images')
diffusion_logs_dir = os.path.join(diffusion_path, 'logs')
diffusion_plots_dir = os.path.join(diffusion_path, 'plots')
fid_results_dir = os.path.join(diffusion_path, 'fid_results')
gridsearch_dir = os.path.join(diffusion_path, 'gridsearch_results')

for dir_path in [diffusion_model_dir, diffusion_images_dir, diffusion_logs_dir,
                 diffusion_plots_dir, fid_results_dir, gridsearch_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Dataset transformations
def get_data_transform(augment_strength='medium'):
    """Get data transforms with different augmentation strengths"""
    if augment_strength == 'light':
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.2),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    elif augment_strength == 'medium':
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.RandomRotation(5),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    elif augment_strength == 'heavy':
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    else:  # none
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

# ==============================================
# 1. Enhanced Diffusion Process with Multiple Schedules
# ==============================================

class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02,
                 img_size=28, device="cuda", schedule_type='cosine'):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size
        self.device = device
        self.schedule_type = schedule_type

        # Different variance schedules
        if schedule_type == 'cosine':
            self.beta = self._cosine_schedule().to(device)
        elif schedule_type == 'linear':
            self.beta = self._linear_schedule().to(device)
        elif schedule_type == 'quadratic':
            self.beta = self._quadratic_schedule().to(device)
        else:
            self.beta = self._cosine_schedule().to(device)

        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def _cosine_schedule(self, s=0.008):
        """Cosine variance schedule"""
        steps = self.noise_steps + 1
        x = torch.linspace(0, self.noise_steps, steps)
        alphas_cumprod = torch.cos(((x / self.noise_steps) + s) / (1 + s) * pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0, 0.999)

    def _linear_schedule(self):
        """Linear variance schedule"""
        return torch.linspace(self.beta_start, self.beta_end, self.noise_steps)

    def _quadratic_schedule(self):
        """Quadratic variance schedule"""
        return torch.linspace(self.beta_start**0.5, self.beta_end**0.5, self.noise_steps) ** 2

    def noise_images(self, x, t):
        """Add noise to images at timestep t"""
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

    def sample_timesteps(self, n):
        """Sample random timesteps for training"""
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model, n, ddim_steps=50):
        """Sample images from model with accelerated DDIM"""
        model.eval()
        with torch.no_grad():
            if ddim_steps < self.noise_steps:
                step_size = self.noise_steps // ddim_steps
                timesteps = list(range(0, self.noise_steps, step_size))[:ddim_steps]
                timesteps = list(reversed(timesteps))
            else:
                timesteps = list(reversed(range(1, self.noise_steps)))

            x = torch.randn((n, 3, self.img_size, self.img_size)).to(self.device)

            for i, timestep in enumerate(tqdm(timesteps, desc="Sampling", leave=False)):
                t = torch.full((n,), timestep, dtype=torch.long, device=self.device)
                predicted_noise = model(x, t)
                alpha_t = self.alpha_hat[timestep]
                if i < len(timesteps) - 1:
                    alpha_prev = self.alpha_hat[timesteps[i + 1]]
                else:
                    alpha_prev = torch.tensor(1.0).to(self.device)

                pred_x0 = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
                pred_x0 = torch.clamp(pred_x0, -1, 1)

                dir_xt = torch.sqrt(1 - alpha_prev) * predicted_noise
                x = torch.sqrt(alpha_prev) * pred_x0 + dir_xt

        model.train()
        x = (x.clamp(-1, 1) + 1) / 2
        x = (x * 255).type(torch.uint8)
        return x

# ==============================================
# 2. Enhanced U-Net Architecture with Variable Complexity
# ==============================================

class AttentionBlock(nn.Module):
    """Self-attention block"""
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)

        q = q.view(B, C, H * W).transpose(1, 2)
        k = k.view(B, C, H * W)
        v = v.view(B, C, H * W).transpose(1, 2)

        scale = 1 / (C ** 0.5)
        attn = torch.softmax(torch.bmm(q, k) * scale, dim=-1)

        out = torch.bmm(attn, v).transpose(1, 2).view(B, C, H, W)
        out = self.proj(out) + x
        return out

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False, dropout=0.0):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels

        layers = [
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(min(8, mid_channels), mid_channels),
            nn.GELU(),
        ]

        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        layers.extend([
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_channels), out_channels),
        ])

        self.double_conv = nn.Sequential(*layers)

        if self.residual and in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        else:
            self.residual_conv = nn.Identity()

    def forward(self, x):
        if self.residual:
            return F.gelu(self.residual_conv(x) + self.double_conv(x))
        else:
            return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256, dropout=0.0):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True, dropout=dropout),
            DoubleConv(in_channels, out_channels, dropout=dropout),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, t):
        x = self.maxpool_conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256, dropout=0.0):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = nn.Sequential(
            DoubleConv(in_channels, in_channels, residual=True, dropout=dropout),
            DoubleConv(in_channels, out_channels, in_channels // 2, dropout=dropout),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, skip_x, t):
        x = self.up(x)
        diffY = skip_x.size()[2] - x.size()[2]
        diffX = skip_x.size()[3] - x.size()[3]

        x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                      diffY // 2, diffY - diffY // 2])
        x = torch.cat([skip_x, x], dim=1)
        x = self.conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class EnhancedUNet(nn.Module):
    def __init__(self, c_in=3, c_out=3, time_dim=256, device="cuda",
                 base_channels=64, channel_multipliers=[1, 2, 4, 4],
                 dropout=0.0, use_attention=True):
        super().__init__()
        self.time_dim = time_dim
        self.device = device
        self.base_channels = base_channels

        # Calculate channel sizes
        channels = [base_channels * m for m in channel_multipliers]

        # Encoder
        self.inc = DoubleConv(c_in, channels[0], dropout=dropout)
        self.down1 = Down(channels[0], channels[1], time_dim, dropout)
        self.down2 = Down(channels[1], channels[2], time_dim, dropout)
        self.down3 = Down(channels[2], channels[3], time_dim, dropout)

        # Bottleneck
        self.bot1 = DoubleConv(channels[3], channels[3] * 2, dropout=dropout)
        if use_attention:
            self.bot_attn = AttentionBlock(channels[3] * 2)
        else:
            self.bot_attn = nn.Identity()
        self.bot2 = DoubleConv(channels[3] * 2, channels[3], dropout=dropout)

        # Decoder
        self.up1 = Up(channels[3] * 2, channels[2], time_dim, dropout)
        self.up2 = Up(channels[2] + channels[1], channels[1], time_dim, dropout)
        self.up3 = Up(channels[1] + channels[0], channels[0], time_dim, dropout)
        self.outc = nn.Conv2d(channels[0], c_out, kernel_size=1)

    def pos_encoding(self, t, channels):
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, channels, 2, device=self.device).float() / channels)
        )
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc

    def forward(self, x, t):
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)

        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1, t)
        x3 = self.down2(x2, t)
        x4 = self.down3(x3, t)

        # Bottleneck
        x4 = self.bot1(x4)
        x4 = self.bot_attn(x4)
        x4 = self.bot2(x4)

        # Decoder
        x = self.up1(x4, x3, t)
        x = self.up2(x, x2, t)
        x = self.up3(x, x1, t)
        output = self.outc(x)
        return output

# ==============================================
# 3. FID Calculation (Simplified)
# ==============================================

class InceptionV3Features(nn.Module):
    def __init__(self):
        super().__init__()
        inception = inception_v3(pretrained=True)
        inception.eval()

        self.features = nn.Sequential(
            inception.Conv2d_1a_3x3,
            inception.Conv2d_2a_3x3,
            inception.Conv2d_2b_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Conv2d_3b_1x1,
            inception.Conv2d_4a_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Mixed_5b,
            inception.Mixed_5c,
            inception.Mixed_5d,
            inception.Mixed_6a,
            inception.Mixed_6b,
            inception.Mixed_6c,
            inception.Mixed_6d,
            inception.Mixed_6e,
            inception.Mixed_7a,
            inception.Mixed_7b,
            inception.Mixed_7c,
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
        x = (x + 1) / 2
        x = x * 255

        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device) * 255
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device) * 255
        x = (x - mean) / std

        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)

        with torch.no_grad():
            features = self.features(x)
            features = features.view(features.size(0), -1)
        return features

def calculate_fid(real_features, fake_features):
    """Calculate FID between real and generated features"""
    mu_real = np.mean(real_features, axis=0)
    mu_fake = np.mean(fake_features, axis=0)

    sigma_real = np.cov(real_features, rowvar=False)
    sigma_fake = np.cov(fake_features, rowvar=False)

    eps = 1e-6
    sigma_real += eps * np.eye(sigma_real.shape[0])
    sigma_fake += eps * np.eye(sigma_fake.shape[0])

    diff = mu_real - mu_fake
    covmean = sqrtm(sigma_real.dot(sigma_fake))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

def extract_features_from_dataset(dataset, feature_extractor, batch_size=64, max_samples=1000):
    """Extract features from dataset"""
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    features = []
    feature_extractor.eval()
    count = 0

    with torch.no_grad():
        for images, _ in tqdm(dataloader, desc="Extracting real features", leave=False):
            if max_samples and count >= max_samples:
                break
            images = images.to(device)
            batch_features = feature_extractor(images)
            features.append(batch_features.cpu().numpy())
            count += images.size(0)

    return np.concatenate(features, axis=0)

def extract_features_from_generated(generated_images, feature_extractor):
    """Extract features from generated images"""
    features = []
    feature_extractor.eval()

    with torch.no_grad():
        for i in tqdm(range(0, len(generated_images), 64), desc="Extracting generated features", leave=False):
            batch = generated_images[i:i+64].float().to(device)
            batch_features = feature_extractor(batch)
            features.append(batch_features.cpu().numpy())

    return np.concatenate(features, axis=0)

# ==============================================
# 4. Training Function with Hyperparameters
# ==============================================

def train_diffusion_with_params(config, dataset, experiment_name="exp"):
    """Train diffusion model with specific hyperparameters"""

    # Extract hyperparameters
    lr = config['lr']
    batch_size = config['batch_size']
    epochs = config['epochs']
    noise_steps = config['noise_steps']
    base_channels = config['base_channels']
    dropout = config['dropout']
    schedule_type = config['schedule_type']
    use_attention = config['use_attention']
    channel_multipliers = config['channel_multipliers']
    weight_decay = config['weight_decay']

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                           num_workers=2, pin_memory=True)

    # Initialize model and diffusion
    model = EnhancedUNet(
        device=device,
        base_channels=base_channels,
        channel_multipliers=channel_multipliers,
        dropout=dropout,
        use_attention=use_attention
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    diffusion = Diffusion(
        noise_steps=noise_steps,
        img_size=28,
        device=device,
        schedule_type=schedule_type
    )

    # Training loop
    model.train()
    train_losses = []
    best_loss = float('inf')

    for epoch in range(epochs):
        epoch_loss = 0
        progress_bar = tqdm(dataloader, desc=f"{experiment_name} - Epoch {epoch+1}/{epochs}", leave=False)

        for batch_idx, (images, _) in enumerate(progress_bar):
            images = images.to(device)
            t = diffusion.sample_timesteps(images.shape[0]).to(device)
            x_t, noise = diffusion.noise_images(images, t)
            predicted_noise = model(x_t, t)
            loss = F.mse_loss(predicted_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        scheduler.step()
        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)

        if avg_loss < best_loss:
            best_loss = avg_loss

    return model, diffusion, best_loss, train_losses

# ==============================================
# 5. Grid Search Implementation
# ==============================================

def run_grid_search(dataset, quick_mode=False):
    """Run grid search over hyperparameters"""

    if quick_mode:
        # Quick grid search for testing
        param_grid = {
            'lr': [1e-4, 2e-4],
            'batch_size': [64, 128],
            'epochs': [30],
            'noise_steps': [1000],
            'base_channels': [32, 64],
            'dropout': [0.0, 0.1],
            'schedule_type': ['cosine'],
            'use_attention': [True],
            'channel_multipliers': [[1, 2, 4, 4]],
            'weight_decay': [1e-4]
        }
    else:
        # Full grid search
        param_grid = {
            'lr': [5e-5, 1e-4, 2e-4, 3e-4],
            'batch_size': [64, 128, 256],
            'epochs': [80, 120, 150],
            'noise_steps': [500, 1000, 1500],
            'base_channels': [32, 64, 96],
            'dropout': [0.0, 0.1, 0.2],
            'schedule_type': ['cosine', 'linear', 'quadratic'],
            'use_attention': [True, False],
            'channel_multipliers': [[1, 2, 4, 4], [1, 2, 4, 8], [1, 1, 2, 4]],
            'weight_decay': [1e-5, 1e-4, 5e-4]
        }

    # Generate all combinations
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    all_combinations = list(product(*param_values))

    print(f"Total combinations to test: {len(all_combinations)}")

    results = []
    best_fid = float('inf')
    best_config = None
    best_model = None
    best_diffusion = None

    # Feature extractor for FID calculation
    feature_extractor = InceptionV3Features().to(device)
    real_features = extract_features_from_dataset(dataset, feature_extractor, max_samples=800)

    for i, param_combination in enumerate(all_combinations):
        config = dict(zip(param_names, param_combination))
        experiment_name = f"exp_{i+1:03d}"

        print(f"\n{'='*60}")
        print(f"Experiment {i+1}/{len(all_combinations)}: {experiment_name}")
        print(f"Config: {config}")
        print(f"{'='*60}")

        try:
            # Train model
            model, diffusion, best_loss, train_losses = train_diffusion_with_params(
                config, dataset, experiment_name
            )

            # Generate samples for FID
            print("Generating samples for FID calculation...")
            num_samples = 500  # Reduced for speed
            samples = diffusion.sample(model, n=num_samples, ddim_steps=20)
            samples = samples.float() / 255.0

            # Calculate FID
            fake_features = extract_features_from_generated(samples, feature_extractor)

            # Use subset for speed
            sample_size = min(400, len(real_features), len(fake_features))
            real_subset = real_features[:sample_size]
            fake_subset = fake_features[:sample_size]

            fid_score = calculate_fid(real_subset, fake_subset)

            # Store results
            result = {
                'experiment': experiment_name,
                'config': config,
                'best_loss': best_loss,
                'final_loss': train_losses[-1],
                'fid_score': fid_score,
                'model_params': sum(p.numel() for p in model.parameters())
            }

            results.append(result)

            print(f"✅ Best Loss: {best_loss:.6f}")
            print(f"✅ FID Score: {fid_score:.4f}")
            print(f"✅ Model Parameters: {result['model_params']:,}")

            # Update best model
            if fid_score < best_fid:
                best_fid = fid_score
                best_config = config.copy()
                best_model = model
                best_diffusion = diffusion

                # Save best model
                torch.save({
                    'config': config,
                    'model_state_dict': model.state_dict(),
                    'fid_score': fid_score,
                    'best_loss': best_loss
                }, os.path.join(diffusion_model_dir, 'best_gridsearch_model.pth'))

                print(f"🏆 NEW BEST FID: {fid_score:.4f}")

            # Save intermediate results
            with open(os.path.join(gridsearch_dir, 'grid_search_results.json'), 'w') as f:
                json.dump(results, f, indent=2)

        except Exception as e:
            print(f"❌ Error in experiment {experiment_name}: {str(e)}")
            result = {
                'experiment': experiment_name,
                'config': config,
                'error': str(e),
                'fid_score': float('inf')
            }
            results.append(result)

        # Clean up GPU memory
        torch.cuda.empty_cache()

    return results, best_config, best_model, best_diffusion, best_fid

# ==============================================
# 6. Analysis and Visualization
# ==============================================

def analyze_grid_search_results(results):
    """Analyze and visualize grid search results"""

    # Filter successful experiments
    successful_results = [r for r in results if 'error' not in r and r['fid_score'] != float('inf')]

    if not successful_results:
        print("No successful experiments found!")
        return

    # Convert to DataFrame-like structure for analysis
    print(f"\n{'='*60}")
    print("GRID SEARCH ANALYSIS")
    print(f"{'='*60}")
    print(f"Total experiments: {len(results)}")
    print(f"Successful experiments: {len(successful_results)}")

    # Sort by FID score
    successful_results.sort(key=lambda x: x['fid_score'])

    # Top 10 results
    print(f"\n🏆 TOP 10 BEST FID SCORES:")
    print("-" * 80)
    for i, result in enumerate(successful_results[:10]):
        print(f"{i+1:2d}. FID: {result['fid_score']:6.2f} | Loss: {result['best_loss']:.6f} | Exp: {result['experiment']}")

    # Best configuration details
    best_result = successful_results[0]
    print(f"\n🥇 BEST CONFIGURATION:")
    print("-" * 40)
    for key, value in best_result['config'].items():
        print(f"{key:20s}: {value}")
    print(f"{'FID Score':20s}: {best_result['fid_score']:.4f}")
    print(f"{'Best Loss':20s}: {best_result['best_loss']:.6f}")
    print(f"{'Model Params':20s}: {best_result['model_params']:,}")

    # Parameter importance analysis
    print(f"\n📊 PARAMETER ANALYSIS:")
    print("-" * 40)

    # Analyze impact of different parameters
    param_impact = {}
    for param in ['lr', 'batch_size', 'base_channels', 'dropout', 'schedule_type', 'use_attention']:
        if param in successful_results[0]['config']:
            param_values = {}
            for result in successful_results:
                value = result['config'][param]
                if value not in param_values:
                    param_values[value] = []
                param_values[value].append(result['fid_score'])

            # Calculate average FID for each parameter value
            avg_fids = {k: np.mean(v) for k, v in param_values.items()}
            best_value = min(avg_fids.keys(), key=lambda x: avg_fids[x])
            param_impact[param] = {
                'best_value': best_value,
                'best_avg_fid': avg_fids[best_value],
                'all_values': avg_fids
            }

    for param, info in param_impact.items():
        print(f"{param:15s}: Best = {info['best_value']} (avg FID: {info['best_avg_fid']:.2f})")

    # Create visualization plots
    create_analysis_plots(successful_results, param_impact)

    return successful_results, param_impact

def create_analysis_plots(results, param_impact):
    """Create visualization plots for grid search analysis"""

    # Plot 1: FID vs Loss scatter
    plt.figure(figsize=(15, 10))

    plt.subplot(2, 3, 1)
    fid_scores = [r['fid_score'] for r in results]
    best_losses = [r['best_loss'] for r in results]
    plt.scatter(best_losses, fid_scores, alpha=0.6)
    plt.xlabel('Best Training Loss')
    plt.ylabel('FID Score')
    plt.title('FID vs Training Loss')
    plt.grid(True, alpha=0.3)

    # Plot 2: Learning Rate Impact
    plt.subplot(2, 3, 2)
    if 'lr' in param_impact:
        lrs = list(param_impact['lr']['all_values'].keys())
        avg_fids = list(param_impact['lr']['all_values'].values())
        plt.bar(range(len(lrs)), avg_fids)
        plt.xticks(range(len(lrs)), [f'{lr:.0e}' for lr in lrs], rotation=45)
        plt.ylabel('Average FID Score')
        plt.title('Learning Rate Impact')
        plt.grid(True, alpha=0.3)

    # Plot 3: Batch Size Impact
    plt.subplot(2, 3, 3)
    if 'batch_size' in param_impact:
        batch_sizes = list(param_impact['batch_size']['all_values'].keys())
        avg_fids = list(param_impact['batch_size']['all_values'].values())
        plt.bar(batch_sizes, avg_fids)
        plt.xlabel('Batch Size')
        plt.ylabel('Average FID Score')
        plt.title('Batch Size Impact')
        plt.grid(True, alpha=0.3)

    # Plot 4: Base Channels Impact
    plt.subplot(2, 3, 4)
    if 'base_channels' in param_impact:
        channels = list(param_impact['base_channels']['all_values'].keys())
        avg_fids = list(param_impact['base_channels']['all_values'].values())
        plt.bar(channels, avg_fids)
        plt.xlabel('Base Channels')
        plt.ylabel('Average FID Score')
        plt.title('Base Channels Impact')
        plt.grid(True, alpha=0.3)

    # Plot 5: FID Distribution
    plt.subplot(2, 3, 5)
    plt.hist(fid_scores, bins=20, alpha=0.7, edgecolor='black')
    plt.xlabel('FID Score')
    plt.ylabel('Frequency')
    plt.title('FID Score Distribution')
    plt.grid(True, alpha=0.3)

    # Plot 6: Top 10 Experiments
    plt.subplot(2, 3, 6)
    top_10 = sorted(results, key=lambda x: x['fid_score'])[:10]
    exp_names = [r['experiment'] for r in top_10]
    fid_scores_top = [r['fid_score'] for r in top_10]
    plt.barh(range(len(exp_names)), fid_scores_top)
    plt.yticks(range(len(exp_names)), exp_names)
    plt.xlabel('FID Score')
    plt.title('Top 10 Best Experiments')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(diffusion_plots_dir, 'grid_search_analysis.png'),
                dpi=300, bbox_inches='tight')
    plt.close()

# ==============================================
# 7. Final Evaluation with Best Model
# ==============================================

def final_evaluation(best_model, best_diffusion, dataset, num_samples=2000):
    """Final evaluation of the best model"""

    print(f"\n{'='*60}")
    print("FINAL EVALUATION WITH BEST MODEL")
    print(f"{'='*60}")

    # Generate more samples
    print(f"Generating {num_samples} samples for final evaluation...")

    batch_size = 200
    all_generated = []

    for i in tqdm(range(0, num_samples, batch_size), desc="Generating final samples"):
        current_batch = min(batch_size, num_samples - i)
        samples = best_diffusion.sample(best_model, n=current_batch, ddim_steps=50)
        samples = samples.float() / 255.0
        all_generated.append(samples.cpu())
        torch.cuda.empty_cache()

    all_generated = torch.cat(all_generated, dim=0)

    # Save sample grid
    final_dir = os.path.join(diffusion_images_dir, 'final_evaluation')
    os.makedirs(final_dir, exist_ok=True)

    # Create multiple grids
    for grid_size in [16, 64, 144]:
        if len(all_generated) >= grid_size:
            grid = utils.make_grid(all_generated[:grid_size],
                                 nrow=int(np.sqrt(grid_size)),
                                 padding=2)
            save_image(grid, os.path.join(final_dir, f'final_grid_{grid_size}.png'))

    # Calculate final FID with multiple runs
    print("Calculating final FID score...")
    feature_extractor = InceptionV3Features().to(device)

    # Extract real features
    real_features = extract_features_from_dataset(dataset, feature_extractor, max_samples=1500)

    # Extract generated features
    fake_features = extract_features_from_generated(all_generated[:1500], feature_extractor)

    # Calculate FID multiple times for stability
    fid_scores = []
    for run in range(5):
        sample_size = min(1000, len(real_features), len(fake_features))
        real_idx = np.random.choice(len(real_features), sample_size, replace=False)
        fake_idx = np.random.choice(len(fake_features), sample_size, replace=False)

        fid_score = calculate_fid(real_features[real_idx], fake_features[fake_idx])
        fid_scores.append(fid_score)

    final_fid_mean = np.mean(fid_scores)
    final_fid_std = np.std(fid_scores)

    print(f"\n🎯 FINAL RESULTS:")
    print(f"📊 Final FID Score: {final_fid_mean:.4f} ± {final_fid_std:.4f}")
    print(f"📊 Individual FID Scores: {[f'{s:.2f}' for s in fid_scores]}")

    # Save final results
    final_results = {
        'final_fid_mean': final_fid_mean,
        'final_fid_std': final_fid_std,
        'fid_scores': fid_scores,
        'num_samples_generated': len(all_generated),
        'num_samples_evaluated': sample_size
    }

    with open(os.path.join(fid_results_dir, 'final_evaluation.json'), 'w') as f:
        json.dump(final_results, f, indent=2)

    return final_results, all_generated

# ==============================================
# 8. Main Execution Function
# ==============================================

def main(quick_mode=False, augment_strength='medium'):
    """Main execution function"""

    print(f"🚀 Starting Diffusion Model Grid Search")
    print(f"Using device: {device}")
    print(f"Quick mode: {quick_mode}")
    print(f"Augmentation strength: {augment_strength}")

    # Load dataset with specified augmentation
    data_transform = get_data_transform(augment_strength)

    train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
    val_dataset = BloodMNIST(split='val', transform=data_transform, download=True)
    test_dataset = BloodMNIST(split='test', transform=data_transform, download=True)
    full_dataset = torch.utils.data.ConcatDataset([train_dataset, val_dataset, test_dataset])

    print(f"Dataset size: {len(full_dataset)} images")

    # Run grid search
    print(f"\n🔍 Starting Grid Search...")
    results, best_config, best_model, best_diffusion, best_fid = run_grid_search(
        full_dataset, quick_mode=quick_mode
    )

    # Analyze results
    print(f"\n📈 Analyzing Results...")
    successful_results, param_impact = analyze_grid_search_results(results)

    # Final evaluation
    if best_model is not None:
        print(f"\n🎯 Running Final Evaluation...")
        final_results, final_generated = final_evaluation(
            best_model, best_diffusion, full_dataset,
            num_samples=1000 if quick_mode else 2000
        )

        # Create comprehensive summary
        summary = {
            'grid_search_summary': {
                'total_experiments': len(results),
                'successful_experiments': len(successful_results),
                'best_fid_from_search': best_fid,
                'best_config': best_config
            },
            'final_evaluation': final_results,
            'parameter_analysis': param_impact,
            'dataset_info': {
                'total_samples': len(full_dataset),
                'augmentation': augment_strength
            }
        }

        # Save comprehensive summary
        with open(os.path.join(diffusion_path, 'comprehensive_summary.json'), 'w') as f:
            json.dump(summary, f, indent=2)

        # Print final summary
        print(f"\n{'='*70}")
        print("🎉 EXPERIMENT COMPLETED SUCCESSFULLY! 🎉")
        print(f"{'='*70}")
        print(f"📊 Best Grid Search FID: {best_fid:.4f}")
        print(f"📊 Final Evaluation FID: {final_results['final_fid_mean']:.4f} ± {final_results['final_fid_std']:.4f}")
        print(f"📁 Results saved in: {diffusion_path}")
        print(f"🏆 Best Configuration:")
        for key, value in best_config.items():
            print(f"   {key}: {value}")
        print(f"{'='*70}")

        return summary, best_model, best_diffusion

    else:
        print("❌ No successful models found in grid search!")
        return None, None, None

# ==============================================
# 9. Execute the Grid Search
# ==============================================

if __name__ == "__main__":
    # Run the grid search
    # Set quick_mode=True for faster testing, False for comprehensive search
    summary, best_model, best_diffusion = main(quick_mode=True, augment_strength='medium')

    # If you want to run a comprehensive search, uncomment the line below:
    # summary, best_model, best_diffusion = main(quick_mode=False, augment_strength='medium')

🚀 Starting Diffusion Model Grid Search
Using device: cuda
Quick mode: True
Augmentation strength: medium
Dataset size: 17092 images

🔍 Starting Grid Search...
Total combinations to test: 16



Experiment 1/16: exp_001
Config: {'lr': 0.0001, 'batch_size': 64, 'epochs': 30, 'noise_steps': 1000, 'base_channels': 32, 'dropout': 0.0, 'schedule_type': 'cosine', 'use_attention': True, 'channel_multipliers': [1, 2, 4, 4], 'weight_decay': 0.0001}


Generating samples for FID calculation...


✅ Best Loss: 0.049900
✅ FID Score: 185.0055
✅ Model Parameters: 5,434,179
🏆 NEW BEST FID: 185.0055

Experiment 2/16: exp_002
Config: {'lr': 0.0001, 'batch_size': 64, 'epochs': 30, 'noise_steps': 1000, 'base_channels': 32, 'dropout': 0.1, 'schedule_type': 'cosine', 'use_attention': True, 'channel_multipliers': [1, 2, 4, 4], 'weight_decay': 0.0001}


Generating samples for FID calculation...


...

Generating samples for FID calculation...


✅ Best Loss: 0.052785
✅ FID Score: 109.7745
✅ Model Parameters: 21,438,083

📈 Analyzing Results...

GRID SEARCH ANALYSIS
Total experiments: 16
Successful experiments: 16

🏆 TOP 10 BEST FID SCORES:
--------------------------------------------------------------------------------
 1. FID: 107.34 | Loss: 0.047675 | Exp: exp_012
 2. FID: 109.77 | Loss: 0.052785 | Exp: exp_016
 3. FID: 112.97 | Loss: 0.052956 | Exp: exp_004
 4. FID: 116.75 | Loss: 0.058651 | Exp: exp_008
 5. FID: 132.78 | Loss: 0.062806 | Exp: exp_014
 6. FID: 136.72 | Loss: 0.053444 | Exp: exp_010
 7. FID: 137.13 | Loss: 0.040744 | Exp: exp_011
 8. FID: 137.88 | Loss: 0.044197 | Exp: exp_003
 9. FID: 142.48 | Loss: 0.063485 | Exp: exp_002
10. FID: 148.39 | Loss: 0.043468 | Exp: exp_015

🥇 BEST CONFIGURATION:
----------------------------------------
lr                  : 0.0002
batch_size          : 64
epochs              : 30
noise_steps         : 1000
base_channels       : 64
dropout             : 0.1
schedule_type       :

Generating final samples: 100%|██████████| 5/5 [00:13<00:00,  2.73s/it]


Calculating final FID score...



🎯 FINAL RESULTS:
📊 Final FID Score: 86.1458 ± 0.2069
📊 Individual FID Scores: ['86.17', '85.89', '86.00', '86.18', '86.50']

🎉 EXPERIMENT COMPLETED SUCCESSFULLY! 🎉
📊 Best Grid Search FID: 107.3411
📊 Final Evaluation FID: 86.1458 ± 0.2069
📁 Results saved in: diffusion_gridsearch_20250527-131909
🏆 Best Configuration:
   lr: 0.0002
   batch_size: 64
   epochs: 30
   noise_steps: 1000
   base_channels: 64
   dropout: 0.1
   schedule_type: cosine
   use_attention: True
   channel_multipliers: [1, 2, 4, 4]
   weight_decay: 0.0001


## Ponto 4 - Treino Completo do Modelo Final

### Configuração Otimizada
- Utilização da melhor combinação de hiperparâmetros identificada via Grid Search.
- Arquitetura e parâmetros ajustados com base nos melhores resultados obtidos anteriormente.

### Treinamento Estendido
- Aumento do número de épocas de 30 para 150, permitindo uma melhor convergência do modelo.
- Monitoramento constante das métricas de validação para evitar overfitting.

### Recursos Avançados
- Cosine Annealing: Scheduler de taxa de aprendizado que diminui suavemente ao longo do tempo.
- Gradient Clipping: Limitação da magnitude dos gradientes para estabilizar o treinamento.
- Checkpointing: Salvamento automático do melhor modelo com base na métrica de avaliação.

### Avaliação Rigorosa
- FID Score (Fréchet Inception Distance) como métrica principal de avaliação da qualidade das imagens geradas.
- Execução do treinamento e avaliação em múltiplas execuções com seeds diferentes para garantir robustez e reprodutibilidade.

### Documentação dos Resultados
- Registro detalhado de:
  - Hiperparâmetros utilizados
  - Gráficos de evolução das métricas (treino e validação)
  - Tabelas comparativas entre execuções
  - Versão final do modelo treinado


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, utils
from torchvision.utils import save_image
from medmnist import BloodMNIST
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import os
import time
from torch.utils.tensorboard import SummaryWriter
from math import sqrt, cos, pi
from scipy.linalg import sqrtm
from torchvision.models import inception_v3
import shutil
import json
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Verify Google Drive path
drive_path = '/content/drive/MyDrive/'
if not os.path.exists(drive_path):
    raise FileNotFoundError(f"Google Drive path not found: {drive_path}")

# Basic configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
timestamp = time.strftime("%Y%m%d-%H%M%S")

# Create project directory
diffusion_path = os.path.join(drive_path, f'diffusion_bestconfig_{timestamp}')
os.makedirs(diffusion_path, exist_ok=True)

# Subdirectories
diffusion_model_dir = os.path.join(diffusion_path, 'models')
diffusion_images_dir = os.path.join(diffusion_path, 'generated_images')
diffusion_logs_dir = os.path.join(diffusion_path, 'logs')
diffusion_plots_dir = os.path.join(diffusion_path, 'plots')
fid_results_dir = os.path.join(diffusion_path, 'fid_results')

for dir_path in [diffusion_model_dir, diffusion_images_dir, diffusion_logs_dir,
                 diffusion_plots_dir, fid_results_dir]:
    os.makedirs(dir_path, exist_ok=True)

# Test write permissions
test_file = os.path.join(diffusion_path, 'test_write.txt')
try:
    with open(test_file, 'w') as f:
        f.write("Test write operation")
    os.remove(test_file)
    print(f"✓ Write permissions confirmed in {diffusion_path}")
except Exception as e:
    print(f"✗ Failed to write to Google Drive: {e}")
    raise

print("\nDirectory structure created:")
print(f"• Models: {diffusion_model_dir}")
print(f"• Images: {diffusion_images_dir}")
print(f"• Logs: {diffusion_logs_dir}")
print(f"• Plots: {diffusion_plots_dir}")
print(f"• FID Results: {fid_results_dir}\n")

# Dataset transformations
def get_data_transform(augment_strength='medium'):
    """Get data transforms with different augmentation strengths"""
    if augment_strength == 'light':
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.2),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    elif augment_strength == 'medium':
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.3),
            transforms.RandomRotation(5),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    elif augment_strength == 'heavy':
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])
    else:  # none
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

# ==============================================
# Enhanced Diffusion Process with Cosine Schedule
# ==============================================

class Diffusion:
    def __init__(self, noise_steps=1000, beta_start=1e-4, beta_end=0.02,
                 img_size=28, device="cuda", schedule_type='cosine'):
        self.noise_steps = noise_steps
        self.beta_start = beta_start
        self.beta_end = beta_end
        self.img_size = img_size
        self.device = device
        self.schedule_type = schedule_type

        # Cosine variance schedule
        self.beta = self._cosine_schedule().to(device)
        self.alpha = 1. - self.beta
        self.alpha_hat = torch.cumprod(self.alpha, dim=0)

    def _cosine_schedule(self, s=0.008):
        """Cosine variance schedule"""
        steps = self.noise_steps + 1
        x = torch.linspace(0, self.noise_steps, steps)
        alphas_cumprod = torch.cos(((x / self.noise_steps) + s) / (1 + s) * pi / 2) ** 2
        alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
        betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
        return torch.clip(betas, 0, 0.999)

    def noise_images(self, x, t):
        """Add noise to images at timestep t"""
        sqrt_alpha_hat = torch.sqrt(self.alpha_hat[t])[:, None, None, None]
        sqrt_one_minus_alpha_hat = torch.sqrt(1 - self.alpha_hat[t])[:, None, None, None]
        epsilon = torch.randn_like(x)
        return sqrt_alpha_hat * x + sqrt_one_minus_alpha_hat * epsilon, epsilon

    def sample_timesteps(self, n):
        """Sample random timesteps for training"""
        return torch.randint(low=1, high=self.noise_steps, size=(n,))

    def sample(self, model, n, ddim_steps=50):
        """Sample images from model with accelerated DDIM"""
        model.eval()
        with torch.no_grad():
            if ddim_steps < self.noise_steps:
                step_size = self.noise_steps // ddim_steps
                timesteps = list(range(0, self.noise_steps, step_size))[:ddim_steps]
                timesteps = list(reversed(timesteps))
            else:
                timesteps = list(reversed(range(1, self.noise_steps)))

            x = torch.randn((n, 3, self.img_size, self.img_size)).to(self.device)

            for i, timestep in enumerate(tqdm(timesteps, desc="Sampling", leave=False)):
                t = torch.full((n,), timestep, dtype=torch.long, device=self.device)
                predicted_noise = model(x, t)
                alpha_t = self.alpha_hat[timestep]
                if i < len(timesteps) - 1:
                    alpha_prev = self.alpha_hat[timesteps[i + 1]]
                else:
                    alpha_prev = torch.tensor(1.0).to(self.device)

                pred_x0 = (x - torch.sqrt(1 - alpha_t) * predicted_noise) / torch.sqrt(alpha_t)
                pred_x0 = torch.clamp(pred_x0, -1, 1)

                dir_xt = torch.sqrt(1 - alpha_prev) * predicted_noise
                x = torch.sqrt(alpha_prev) * pred_x0 + dir_xt

        model.train()
        x = (x.clamp(-1, 1) + 1) / 2
        x = (x * 255).type(torch.uint8)
        return x

# ==============================================
# Enhanced U-Net Architecture with Best Config
# ==============================================

class AttentionBlock(nn.Module):
    """Self-attention block"""
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.norm = nn.GroupNorm(min(8, channels), channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h)
        q, k, v = qkv.chunk(3, dim=1)

        q = q.view(B, C, H * W).transpose(1, 2)
        k = k.view(B, C, H * W)
        v = v.view(B, C, H * W).transpose(1, 2)

        scale = 1 / (C ** 0.5)
        attn = torch.softmax(torch.bmm(q, k) * scale, dim=-1)

        out = torch.bmm(attn, v).transpose(1, 2).view(B, C, H, W)
        out = self.proj(out) + x
        return out

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None, residual=False, dropout=0.0):
        super().__init__()
        self.residual = residual
        if not mid_channels:
            mid_channels = out_channels

        layers = [
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(min(8, mid_channels), mid_channels),
            nn.GELU(),
        ]

        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        layers.extend([
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(min(8, out_channels), out_channels),
        ])

        self.double_conv = nn.Sequential(*layers)

        if self.residual and in_channels != out_channels:
            self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        else:
            self.residual_conv = nn.Identity()

    def forward(self, x):
        if self.residual:
            return F.gelu(self.residual_conv(x) + self.double_conv(x))
        else:
            return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256, dropout=0.0):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, in_channels, residual=True, dropout=dropout),
            DoubleConv(in_channels, out_channels, dropout=dropout),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, t):
        x = self.maxpool_conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, emb_dim=256, dropout=0.0):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv = nn.Sequential(
            DoubleConv(in_channels, in_channels, residual=True, dropout=dropout),
            DoubleConv(in_channels, out_channels, in_channels // 2, dropout=dropout),
        )

        self.emb_layer = nn.Sequential(
            nn.SiLU(),
            nn.Linear(emb_dim, out_channels),
        )

    def forward(self, x, skip_x, t):
        x = self.up(x)
        diffY = skip_x.size()[2] - x.size()[2]
        diffX = skip_x.size()[3] - x.size()[3]

        x = F.pad(x, [diffX // 2, diffX - diffX // 2,
                      diffY // 2, diffY - diffY // 2])
        x = torch.cat([skip_x, x], dim=1)
        x = self.conv(x)
        emb = self.emb_layer(t)[:, :, None, None].repeat(1, 1, x.shape[-2], x.shape[-1])
        return x + emb

class EnhancedUNet(nn.Module):
    def __init__(self, c_in=3, c_out=3, time_dim=256, device="cuda",
                 base_channels=64, channel_multipliers=[1, 2, 4, 4],
                 dropout=0.1, use_attention=True):
        super().__init__()
        self.time_dim = time_dim
        self.device = device
        self.base_channels = base_channels

        # Calculate channel sizes
        channels = [base_channels * m for m in channel_multipliers]

        # Encoder
        self.inc = DoubleConv(c_in, channels[0], dropout=dropout)
        self.down1 = Down(channels[0], channels[1], time_dim, dropout)
        self.down2 = Down(channels[1], channels[2], time_dim, dropout)
        self.down3 = Down(channels[2], channels[3], time_dim, dropout)

        # Bottleneck
        self.bot1 = DoubleConv(channels[3], channels[3] * 2, dropout=dropout)
        if use_attention:
            self.bot_attn = AttentionBlock(channels[3] * 2)
        else:
            self.bot_attn = nn.Identity()
        self.bot2 = DoubleConv(channels[3] * 2, channels[3], dropout=dropout)

        # Decoder
        self.up1 = Up(channels[3] * 2, channels[2], time_dim, dropout)
        self.up2 = Up(channels[2] + channels[1], channels[1], time_dim, dropout)
        self.up3 = Up(channels[1] + channels[0], channels[0], time_dim, dropout)
        self.outc = nn.Conv2d(channels[0], c_out, kernel_size=1)

    def pos_encoding(self, t, channels):
        inv_freq = 1.0 / (
            10000 ** (torch.arange(0, channels, 2, device=self.device).float() / channels)
        )
        pos_enc_a = torch.sin(t.repeat(1, channels // 2) * inv_freq)
        pos_enc_b = torch.cos(t.repeat(1, channels // 2) * inv_freq)
        pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)
        return pos_enc

    def forward(self, x, t):
        t = t.unsqueeze(-1).type(torch.float)
        t = self.pos_encoding(t, self.time_dim)

        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1, t)
        x3 = self.down2(x2, t)
        x4 = self.down3(x3, t)

        # Bottleneck
        x4 = self.bot1(x4)
        x4 = self.bot_attn(x4)
        x4 = self.bot2(x4)

        # Decoder
        x = self.up1(x4, x3, t)
        x = self.up2(x, x2, t)
        x = self.up3(x, x1, t)
        output = self.outc(x)
        return output

# ==============================================
# Training Function with Best Configuration
# ==============================================

def train_diffusion(dataset, epochs=150):
    """Train diffusion model with best configuration"""

    # Best configuration from grid search
    config = {
        'lr': 2e-4,
        'batch_size': 64,
        'epochs': epochs,
        'noise_steps': 1000,
        'base_channels': 64,
        'dropout': 0.1,
        'schedule_type': 'cosine',
        'use_attention': True,
        'channel_multipliers': [1, 2, 4, 4],
        'weight_decay': 1e-4
    }

    # Save config file
    config_file = os.path.join(diffusion_path, 'config.json')
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2)
    print(f"✓ Configuration saved to {config_file}")

    # Create dataloader
    dataloader = DataLoader(dataset, batch_size=config['batch_size'], shuffle=True,
                           num_workers=2, pin_memory=True)

    # Initialize model and diffusion
    model = EnhancedUNet(
        device=device,
        base_channels=config['base_channels'],
        channel_multipliers=config['channel_multipliers'],
        dropout=config['dropout'],
        use_attention=config['use_attention']
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],
                                weight_decay=config['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    diffusion = Diffusion(
        noise_steps=config['noise_steps'],
        img_size=28,
        device=device,
        schedule_type=config['schedule_type']
    )

    # Training loop
    model.train()
    train_losses = []
    best_loss = float('inf')

    # Tensorboard writer
    writer = SummaryWriter(log_dir=diffusion_logs_dir)

    for epoch in range(epochs):
        epoch_loss = 0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

        for batch_idx, (images, _) in enumerate(progress_bar):
            images = images.to(device)
            t = diffusion.sample_timesteps(images.shape[0]).to(device)
            x_t, noise = diffusion.noise_images(images, t)
            predicted_noise = model(x_t, t)
            loss = F.mse_loss(predicted_noise, noise)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            epoch_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

            # Log batch loss to tensorboard
            writer.add_scalar('Batch Loss', loss.item(), epoch * len(dataloader) + batch_idx)

        scheduler.step()
        avg_loss = epoch_loss / len(dataloader)
        train_losses.append(avg_loss)

        # Log epoch loss to tensorboard
        writer.add_scalar('Epoch Loss', avg_loss, epoch)
        writer.add_scalar('Learning Rate', scheduler.get_last_lr()[0], epoch)

        # Save checkpoint
        if avg_loss < best_loss:
            best_loss = avg_loss
            checkpoint_path = os.path.join(diffusion_model_dir, 'best_model.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
                'config': config
            }, checkpoint_path)
            print(f"✓ New best model saved to {checkpoint_path} (loss: {best_loss:.4f})")

        # Save sample images every 10 epochs
        if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
            model.eval()
            with torch.no_grad():
                samples = diffusion.sample(model, n=16, ddim_steps=50)
                grid = utils.make_grid(samples, nrow=4)
                sample_path = os.path.join(diffusion_images_dir, f'samples_epoch_{epoch+1}.png')
                save_image(grid.float()/255, sample_path)
                print(f"✓ Sample images saved to {sample_path}")
            model.train()

    writer.close()

    # Save final model
    final_model_path = os.path.join(diffusion_model_dir, 'final_model.pth')
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': train_losses[-1],
        'config': config,
        'train_losses': train_losses
    }, final_model_path)
    print(f"✓ Final model saved to {final_model_path}")

    # Plot training loss
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Loss Curve')
    plt.legend()
    plt.grid(True)
    loss_plot_path = os.path.join(diffusion_plots_dir, 'training_loss.png')
    plt.savefig(loss_plot_path)
    plt.close()
    print(f"✓ Training loss plot saved to {loss_plot_path}")

    return model, diffusion, best_loss, train_losses

# ==============================================
# FID Calculation with 10k images
# ==============================================

class InceptionV3Features(nn.Module):
    def __init__(self):
        super().__init__()
        inception = inception_v3(pretrained=True)
        inception.eval()

        self.features = nn.Sequential(
            inception.Conv2d_1a_3x3,
            inception.Conv2d_2a_3x3,
            inception.Conv2d_2b_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Conv2d_3b_1x1,
            inception.Conv2d_4a_3x3,
            nn.MaxPool2d(kernel_size=3, stride=2),
            inception.Mixed_5b,
            inception.Mixed_5c,
            inception.Mixed_5d,
            inception.Mixed_6a,
            inception.Mixed_6b,
            inception.Mixed_6c,
            inception.Mixed_6d,
            inception.Mixed_6e,
            inception.Mixed_7a,
            inception.Mixed_7b,
            inception.Mixed_7c,
            nn.AdaptiveAvgPool2d((1, 1))
        )

    def forward(self, x):
        x = F.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
        x = (x + 1) / 2
        x = x * 255

        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(x.device) * 255
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(x.device) * 255
        x = (x - mean) / std

        if x.shape[1] == 1:
            x = x.repeat(1, 3, 1, 1)

        with torch.no_grad():
            features = self.features(x)
            features = features.view(features.size(0), -1)
        return features

def calculate_fid(real_features, fake_features):
    """Calculate FID between real and generated features"""
    mu_real = np.mean(real_features, axis=0)
    mu_fake = np.mean(fake_features, axis=0)

    sigma_real = np.cov(real_features, rowvar=False)
    sigma_fake = np.cov(fake_features, rowvar=False)

    eps = 1e-6
    sigma_real += eps * np.eye(sigma_real.shape[0])
    sigma_fake += eps * np.eye(sigma_fake.shape[0])

    diff = mu_real - mu_fake
    covmean = sqrtm(sigma_real.dot(sigma_fake))

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    fid = diff.dot(diff) + np.trace(sigma_real + sigma_fake - 2 * covmean)
    return fid

def extract_features_from_dataset(dataset, feature_extractor, batch_size=64, max_samples=None):
    """Extract features from dataset (with support for full dataset)"""
    if max_samples is not None and max_samples < len(dataset):
        # Create a subset sampler if we want to limit samples
        indices = torch.randperm(len(dataset))[:max_samples]
        sampler = torch.utils.data.SubsetRandomSampler(indices)
        dataloader = DataLoader(dataset, batch_size=batch_size, sampler=sampler, num_workers=2)
    else:
        # Use full dataset
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    features = []
    feature_extractor.eval()
    count = 0

    with torch.no_grad():
        for images, _ in tqdm(dataloader, desc="Extracting real features", leave=False):
            images = images.to(device)
            batch_features = feature_extractor(images)
            features.append(batch_features.cpu().numpy())

            if max_samples is not None:
                count += images.size(0)
                if count >= max_samples:
                    break

    return np.concatenate(features, axis=0)

def extract_features_from_generated(generated_images, feature_extractor):
    """Extract features from generated images (optimized for large batches)"""
    features = []
    feature_extractor.eval()
    batch_size = 64  # Process in smaller batches to avoid memory issues

    with torch.no_grad():
        for i in tqdm(range(0, len(generated_images), batch_size),
                     desc="Extracting generated features", leave=False):
            batch = generated_images[i:i+batch_size].float().to(device)
            batch_features = feature_extractor(batch)
            features.append(batch_features.cpu().numpy())

    return np.concatenate(features, axis=0)

def evaluate_model(model, diffusion, dataset):
    """Evaluate the trained model with FID score using 10k images"""
    print("\nEvaluating model with FID score (10k images)...")

    # Feature extractor
    feature_extractor = InceptionV3Features().to(device)

    # Directory to save generated images
    generated_10k_dir = os.path.join(diffusion_path, 'generated_10k')
    os.makedirs(generated_10k_dir, exist_ok=True)

    # Generate 10k samples
    num_samples = 10000
    batch_size = 200  # Process in batches to save memory
    all_generated = []

    print(f"\nGenerating {num_samples} evaluation samples...")
    for i in tqdm(range(0, num_samples, batch_size), desc="Generating 10k samples"):
        current_batch = min(batch_size, num_samples - i)
        samples = diffusion.sample(model, n=current_batch, ddim_steps=50)
        samples = samples.float() / 255.0

        # Save batch of generated images
        for j in range(current_batch):
            img_idx = i + j
            img_path = os.path.join(generated_10k_dir, f'generated_{img_idx:05d}.png')
            save_image(samples[j], img_path)

        all_generated.append(samples.cpu())

    all_generated = torch.cat(all_generated, dim=0)
    print(f"✓ {num_samples} generated images saved to {generated_10k_dir}")

    # Extract generated features
    fake_features = extract_features_from_generated(all_generated, feature_extractor)

    # Extract real features from dataset (we'll sample 10k each time we calculate FID)
    print("\nExtracting features from full dataset...")
    full_real_features = extract_features_from_dataset(dataset, feature_extractor, max_samples=None)

    # Calculate FID multiple times with different random samples
    fid_scores = []
    sample_size = 10000  # We'll use 10k samples each time

    for run in range(5):
        print(f"\nFID calculation run {run + 1}/5")

        # Randomly sample 10k real features
        if len(full_real_features) > sample_size:
            real_idx = np.random.choice(len(full_real_features), sample_size, replace=False)
            current_real_features = full_real_features[real_idx]
        else:
            current_real_features = full_real_features

        # Use all 10k generated features
        current_fake_features = fake_features[:sample_size] if len(fake_features) > sample_size else fake_features

        fid_score = calculate_fid(current_real_features, current_fake_features)
        fid_scores.append(fid_score)

        print(f"Run {run + 1} FID: {fid_score:.2f}")

    final_fid_mean = np.mean(fid_scores)
    final_fid_std = np.std(fid_scores)

    print(f"\nEvaluation Results (10k samples):")
    print(f"• FID Score: {final_fid_mean:.4f} ± {final_fid_std:.4f}")
    print(f"• Individual FID Scores: {[f'{s:.2f}' for s in fid_scores]}")

    # Save results
    results = {
        'fid_mean': final_fid_mean,
        'fid_std': final_fid_std,
        'fid_scores': fid_scores,
        'num_samples': num_samples,
        'generated_images_path': generated_10k_dir,
        'config': {
            'lr': 2e-4,
            'batch_size': 64,
            'epochs': 150,
            'noise_steps': 1000,
            'base_channels': 64,
            'dropout': 0.1,
            'schedule_type': 'cosine',
            'use_attention': True,
            'channel_multipliers': [1, 2, 4, 4],
            'weight_decay': 1e-4
        }
    }

    results_file = os.path.join(fid_results_dir, 'evaluation_results_10k.json')
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Evaluation results saved to {results_file}")

    # Create a zip file of the generated images
    print("\nCreating zip archive of generated images...")
    shutil.make_archive(os.path.join(diffusion_path, 'generated_10k'), 'zip', generated_10k_dir)
    print(f"✓ Zip archive created: {os.path.join(diffusion_path, 'generated_10k.zip')}")

    return results

# ==============================================
# Main Execution
# ==============================================

def main():
    """Main execution function"""
    print(f"\n🚀 Training Diffusion Model with Best Configuration")
    print(f"Using device: {device}")

    # Load dataset with medium augmentation
    data_transform = get_data_transform('medium')

    train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
    val_dataset = BloodMNIST(split='val', transform=data_transform, download=True)
    test_dataset = BloodMNIST(split='test', transform=data_transform, download=True)
    full_dataset = torch.utils.data.ConcatDataset([train_dataset, val_dataset, test_dataset])

    print(f"\nDataset Info:")
    print(f"• Train samples: {len(train_dataset)}")
    print(f"• Val samples: {len(val_dataset)}")
    print(f"• Test samples: {len(test_dataset)}")
    print(f"• Total samples: {len(full_dataset)}")

    # Train model
    print("\n🏋️ Training model for 150 epochs...")
    model, diffusion, best_loss, train_losses = train_diffusion(full_dataset, epochs=150)

    # Evaluate model
    results = evaluate_model(model, diffusion, full_dataset)

    print("\n🎉 Training and evaluation completed!")
    print(f"• Results saved in: {diffusion_path}")
    print(f"• Best training loss: {best_loss:.6f}")
    print(f"• Final FID score: {results['fid_mean']:.4f} ± {results['fid_std']:.4f}")

    # Create a README file with summary
    readme_content = f"""
    # Diffusion Model Training Results

    ## Summary
    - **Timestamp**: {timestamp}
    - **Device**: {device}
    - **Best Loss**: {best_loss:.6f}
    - **Final FID**: {results['fid_mean']:.4f} ± {results['fid_std']:.4f}
    - **Training Time**: {time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))}

    ## Configuration
    ```json
    {json.dumps(results['config'], indent=2)}
    ```
    """

    with open(os.path.join(diffusion_path, 'README.md'), 'w') as f:
        f.write(readme_content)

if __name__ == "__main__":
    start_time = time.time()
    main()
    print(f"\nTotal execution time: {time.strftime('%H:%M:%S', time.gmtime(time.time() - start_time))}")

Mounted at /content/drive
✓ Write permissions confirmed in /content/drive/MyDrive/diffusion_bestconfig_20250528-182156

Directory structure created:
• Models: /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/models
• Images: /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/generated_images
• Logs: /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/logs
• Plots: /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/plots
• FID Results: /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/fid_results


🚀 Training Diffusion Model with Best Configuration
Using device: cuda


100%|██████████| 35.5M/35.5M [00:02<00:00, 13.2MB/s]



Dataset Info:
• Train samples: 11959
• Val samples: 1712
• Test samples: 3421
• Total samples: 17092

🏋️ Training model for 150 epochs...
✓ Configuration saved to /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/config.json


✓ New best model saved to /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/models/best_model.pth (loss: 0.1718)


✓ Sample images saved to /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/generated_images/samples_epoch_1.png


...


Generating 10000 evaluation samples...


Generating 10k samples: 100%|██████████| 50/50 [03:07<00:00,  3.74s/it]


✓ 10000 generated images saved to /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/generated_10k



Extracting features from full dataset...



FID calculation run 1/5
Run 1 FID: 73.30

FID calculation run 2/5
Run 2 FID: 73.52

FID calculation run 3/5
Run 3 FID: 73.57

FID calculation run 4/5
Run 4 FID: 73.31

FID calculation run 5/5
Run 5 FID: 73.44

Evaluation Results (10k samples):
• FID Score: 73.4295 ± 0.1087
• Individual FID Scores: ['73.30', '73.52', '73.57', '73.31', '73.44']
✓ Evaluation results saved to /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/fid_results/evaluation_results_10k.json

Creating zip archive of generated images...
✓ Zip archive created: /content/drive/MyDrive/diffusion_bestconfig_20250528-182156/generated_10k.zip

🎉 Training and evaluation completed!
• Results saved in: /content/drive/MyDrive/diffusion_bestconfig_20250528-182156
• Best training loss: 0.037780
• Final FID score: 73.4295 ± 0.1087

Total execution time: 00:33:00
